<a href="https://colab.research.google.com/github/AlperYildirim1/crt-fourier-transformer-addition/blob/main/Pythia_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## OPTIONAL LEGACY A — MLP-hook causal subspace ablation

This is an older/specialized stream-level experiment. It is **not** part of the numbered main residual pipeline below.

In [ ]:
# ============================================================
# Causal subspace ablation across MLP hooks
# Pythia-6.9B addition
#
# Streams:
#   mlp_input : input to dense_h_to_4h
#   preact    : output of dense_h_to_4h = x @ W_in + b
#   postact   : input to dense_4h_to_h = activation(preact)
#   mlp_out   : output of dense_4h_to_h = residual delta
#
# Intervention:
#   remove learned Fourier/helix subspace at final "=" token
# ============================================================

import os
import json
import random
import warnings
import numpy as np
import torch

from tqdm import tqdm
from sklearn.linear_model import Ridge
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "EleutherAI/pythia-6.9b"

MLP_DIR = "/content/drive/MyDrive/pythia_mlp_full_hooks_addition"

LAYER = 20
A_MAX = 99
B_MAX = 99
BATCH_SIZE = 32
RIDGE_ALPHA = 1.0
SEED = 42

# First set to 1000 for smoke test.
# Then set to None for all baseline-correct examples.
MAX_EVAL_EXAMPLES = 1000

STREAMS_TO_TEST = [
    "mlp_input",
    "preact",
    "postact",
    "mlp_out",
]

INTERVENTIONS = [
    ("no_T2",          [2],             False),
    ("no_T5",          [5],             False),
    ("no_T10",         [10],            False),
    ("no_T2T5",        [2, 5],          False),
    ("no_T2T5T10",     [2, 5, 10],      False),
    ("no_FULL_HELIX",  [2, 5, 10, 100], True),
]


# ============================================================
# SETUP
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


set_seed(SEED)

print_section("LOAD MODEL")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
).eval()

model.config.pad_token_id = tokenizer.pad_token_id

print("model:", MODEL_NAME)
print("device:", model.device)
print("dtype:", next(model.parameters()).dtype)
print("layer:", LAYER)


# ============================================================
# DATASET
# ============================================================

def answer_token_id(n):
    ids = tokenizer(str(int(n)), add_special_tokens=False)["input_ids"]
    if len(ids) != 1:
        return None
    return int(ids[0])


def make_examples():
    rows = []

    for a in range(A_MAX + 1):
        for b in range(B_MAX + 1):
            s = a + b
            tid = answer_token_id(s)
            if tid is None:
                continue

            rows.append({
                "a": int(a),
                "b": int(b),
                "sum": int(s),
                "mod2": int(s % 2),
                "mod5": int(s % 5),
                "mod10": int(s % 10),
                "prompt": f"Output ONLY a number. {a}+{b}=",
                "target_token_id": int(tid),
            })

    return rows


examples = make_examples()
print("examples:", len(examples))


def get_last_positions(attention_mask):
    return attention_mask.sum(dim=1) - 1


def pred_token_to_int(pred_id):
    txt = tokenizer.decode([int(pred_id)]).strip()
    try:
        return int(txt)
    except Exception:
        return None


# ============================================================
# BASELINE-CORRECT FILTER
# ============================================================

@torch.no_grad()
def get_baseline_correct_examples(max_examples=None):
    rows = examples if max_examples is None else examples[:max_examples]
    kept = []

    for start in tqdm(range(0, len(rows), BATCH_SIZE), desc="baseline filter"):
        batch = rows[start:start + BATCH_SIZE]
        prompts = [x["prompt"] for x in batch]

        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(model.device)

        last_pos = get_last_positions(enc["attention_mask"])
        batch_idx = torch.arange(len(batch), device=model.device)

        out = model(**enc, use_cache=False)
        logits = out.logits[batch_idx, last_pos]
        pred_ids = logits.argmax(dim=-1).detach().cpu().tolist()

        for ex, pred_id in zip(batch, pred_ids):
            if int(pred_id) == int(ex["target_token_id"]):
                kept.append(ex)

    return kept


print_section("BASELINE-CORRECT FILTER")
baseline_correct_examples = get_baseline_correct_examples(max_examples=None)

print("baseline correct:", len(baseline_correct_examples), "/", len(examples))
print("baseline exact acc:", len(baseline_correct_examples) / len(examples))

if MAX_EVAL_EXAMPLES is not None:
    eval_examples = baseline_correct_examples[:MAX_EVAL_EXAMPLES]
else:
    eval_examples = baseline_correct_examples

print("eval examples:", len(eval_examples))


# ============================================================
# LEARN SUBSPACE Q FROM SAVED ACTIVATIONS
# ============================================================

def make_fourier_basis(sums, periods, include_linear=False):
    s = sums.astype(np.float64)

    cols = []

    if include_linear:
        z = (s - s.mean()) / (s.std() + 1e-12)
        cols.append(z)

    for T in periods:
        theta = 2.0 * np.pi * s / float(T)
        cols.append(np.cos(theta))
        cols.append(np.sin(theta))

    return np.column_stack(cols)


def load_saved_stream(stream, layer):
    x_path = os.path.join(
        MLP_DIR,
        f"{stream}_layer{layer:02d}_eq_correct_only.pt",
    )
    m_path = os.path.join(
        MLP_DIR,
        f"metadata_layer{layer:02d}_correct_only.jsonl",
    )

    if not os.path.exists(x_path):
        raise FileNotFoundError(f"Missing activation file: {x_path}")
    if not os.path.exists(m_path):
        raise FileNotFoundError(f"Missing metadata file: {m_path}")

    X = torch.load(x_path, map_location="cpu").float().numpy()

    metadata = []
    with open(m_path, "r", encoding="utf-8") as f:
        for line in f:
            metadata.append(json.loads(line))

    if len(metadata) != X.shape[0]:
        raise ValueError(f"metadata mismatch: {len(metadata)} vs {X.shape[0]}")

    sums = np.array([m["sum"] for m in metadata], dtype=np.float64)

    return X, sums


def learn_Q_for_stream(stream, layer, periods, include_linear=False):
    """
    Learn Q directly in the original stream activation space.

    X shape:
      mlp_input/mlp_out: [N, 4096]
      preact/postact:   [N, 16384]

    Q shape:
      [D, k]
    """
    X, sums = load_saved_stream(stream, layer)
    Y = make_fourier_basis(
        sums,
        periods=periods,
        include_linear=include_linear,
    )

    reg = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
    reg.fit(Y, X)

    W = reg.coef_  # [D, K]
    Q, _ = np.linalg.qr(W)

    Q_torch = torch.tensor(Q, dtype=torch.float16, device=model.device)

    print(
        f"learned Q | stream={stream:<9} periods={periods} "
        f"linear={include_linear} X={X.shape} Q={tuple(Q_torch.shape)}"
    )

    return Q_torch


def remove_subspace(x, Q):
    """
    x: [B, D]
    Q: [D, k] orthonormal columns
    """
    return x - (x @ Q) @ Q.T


# ============================================================
# PATCHING HOOKS
# ============================================================

def get_modules(layer):
    mlp = model.gpt_neox.layers[layer].mlp
    return {
        "dense_h_to_4h": mlp.dense_h_to_4h,
        "dense_4h_to_h": mlp.dense_4h_to_h,
    }


def register_ablation_hook(stream, layer, Q):
    """
    Applies x <- x - Proj_Q(x) at final "=" token only.

    stream locations:
      mlp_input : pre-hook on dense_h_to_4h input
      preact    : forward hook on dense_h_to_4h output
      postact   : pre-hook on dense_4h_to_h input
      mlp_out   : forward hook on dense_4h_to_h output
    """
    modules = get_modules(layer)
    dense_h_to_4h = modules["dense_h_to_4h"]
    dense_4h_to_h = modules["dense_4h_to_h"]

    if stream == "mlp_input":

        def pre_hook(module, inputs):
            x = inputs[0].clone()
            x[batch_idx_global, last_pos_global] = remove_subspace(
                x[batch_idx_global, last_pos_global],
                Q,
            )
            return (x,)

        return dense_h_to_4h.register_forward_pre_hook(pre_hook)

    elif stream == "preact":

        def hook(module, inputs, output):
            y = output.clone()
            y[batch_idx_global, last_pos_global] = remove_subspace(
                y[batch_idx_global, last_pos_global],
                Q,
            )
            return y

        return dense_h_to_4h.register_forward_hook(hook)

    elif stream == "postact":

        def pre_hook(module, inputs):
            x = inputs[0].clone()
            x[batch_idx_global, last_pos_global] = remove_subspace(
                x[batch_idx_global, last_pos_global],
                Q,
            )
            return (x,)

        return dense_4h_to_h.register_forward_pre_hook(pre_hook)

    elif stream == "mlp_out":

        def hook(module, inputs, output):
            y = output.clone()
            y[batch_idx_global, last_pos_global] = remove_subspace(
                y[batch_idx_global, last_pos_global],
                Q,
            )
            return y

        return dense_4h_to_h.register_forward_hook(hook)

    else:
        raise ValueError(f"Unknown stream: {stream}")


# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def eval_model_on_rows(rows, stream=None, Q=None):
    """
    If stream/Q are given, applies the causal ablation.
    Rows are baseline-correct examples by default.
    """
    stats = {
        "n": 0,
        "exact": 0,
        "mod2": 0,
        "mod5": 0,
        "mod10": 0,
        "parseable": 0,
    }

    hook = None
    if stream is not None and Q is not None:
        hook = register_ablation_hook(stream, LAYER, Q)

    try:
        for start in tqdm(range(0, len(rows), BATCH_SIZE), desc=f"eval {stream or 'baseline'}"):
            batch = rows[start:start + BATCH_SIZE]
            prompts = [x["prompt"] for x in batch]

            enc = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            ).to(model.device)

            global last_pos_global, batch_idx_global
            last_pos_global = get_last_positions(enc["attention_mask"])
            batch_idx_global = torch.arange(len(batch), device=model.device)

            out = model(**enc, use_cache=False)

            logits = out.logits[batch_idx_global, last_pos_global]
            pred_ids = logits.argmax(dim=-1).detach().cpu().tolist()

            for pred_id, ex in zip(pred_ids, batch):
                stats["n"] += 1

                if int(pred_id) == int(ex["target_token_id"]):
                    stats["exact"] += 1

                pred_num = pred_token_to_int(pred_id)

                if pred_num is None:
                    continue

                stats["parseable"] += 1
                stats["mod2"] += int(pred_num % 2 == ex["mod2"])
                stats["mod5"] += int(pred_num % 5 == ex["mod5"])
                stats["mod10"] += int(pred_num % 10 == ex["mod10"])

    finally:
        if hook is not None:
            hook.remove()

    n = max(stats["n"], 1)

    return {
        "n": stats["n"],
        "exact": stats["exact"] / n,
        "mod2": stats["mod2"] / n,
        "mod5": stats["mod5"] / n,
        "mod10": stats["mod10"] / n,
        "parseable": stats["parseable"] / n,
    }


def print_result_row(name, res):
    print(
        f"{name:<28} "
        f"n={res['n']:<5d} "
        f"exact={res['exact']:.4f} "
        f"mod2={res['mod2']:.4f} "
        f"mod5={res['mod5']:.4f} "
        f"mod10={res['mod10']:.4f} "
        f"parse={res['parseable']:.4f}"
    )


# ============================================================
# RUN
# ============================================================

print_section("BASELINE ON EVAL SET")
baseline_res = eval_model_on_rows(eval_examples, stream=None, Q=None)
print_result_row("baseline", baseline_res)


all_results = []

for stream in STREAMS_TO_TEST:
    print_section(f"STREAM: {stream}")

    # Stream baseline with no intervention is same as global baseline,
    # but printed here for easier comparison.
    print_result_row("baseline", baseline_res)

    for name, periods, include_linear in INTERVENTIONS:
        Q = learn_Q_for_stream(
            stream=stream,
            layer=LAYER,
            periods=periods,
            include_linear=include_linear,
        )

        res = eval_model_on_rows(
            eval_examples,
            stream=stream,
            Q=Q,
        )

        print_result_row(name, res)

        row = {
            "stream": stream,
            "intervention": name,
            "periods": periods,
            "include_linear": include_linear,
            **res,
        }
        all_results.append(row)

        del Q
        torch.cuda.empty_cache()


print_section("SUMMARY TABLE")

print(
    f"{'stream':<10} {'intervention':<18} {'exact':>8} {'mod2':>8} {'mod5':>8} {'mod10':>8} {'parse':>8}"
)
print("-" * 80)

for r in all_results:
    print(
        f"{r['stream']:<10} {r['intervention']:<18} "
        f"{r['exact']:>8.4f} {r['mod2']:>8.4f} {r['mod5']:>8.4f} {r['mod10']:>8.4f} {r['parseable']:>8.4f}"
    )

## OPTIONAL LEGACY B — Single-plane phase-vs-magnitude edit

This is an older/specialized MLP-stream phase/radius edit. It is **not** part of the numbered main residual pipeline below.

In [ ]:
# ============================================================
# Causal phase-vs-magnitude intervention in Fourier plane
# Pythia-6.9B addition
# ============================================================

import os, json, random, warnings
import numpy as np
import torch
from tqdm import tqdm
from sklearn.linear_model import Ridge
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")

MODEL_NAME = "EleutherAI/pythia-6.9b"
MLP_DIR = "/content/drive/MyDrive/pythia_mlp_full_hooks_addition"

LAYER = 20
STREAM = "mlp_out"     # try: "preact", "postact", "mlp_out"
PERIOD = 5             # first try T=5
DELTA = 1              # phase shift by +1 residue
SCALE = 2.0            # magnitude scale test

A_MAX = 99
B_MAX = 99
BATCH_SIZE = 32
RIDGE_ALPHA = 1.0
MAX_EVAL_EXAMPLES = 500
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
).eval()

model.config.pad_token_id = tokenizer.pad_token_id


def answer_token_id(n):
    ids = tokenizer(str(int(n)), add_special_tokens=False)["input_ids"]
    return int(ids[0]) if len(ids) == 1 else None


def make_examples():
    rows = []
    for a in range(A_MAX + 1):
        for b in range(B_MAX + 1):
            s = a + b
            tid = answer_token_id(s)
            if tid is None:
                continue
            rows.append({
                "a": int(a),
                "b": int(b),
                "sum": int(s),
                "mod2": int(s % 2),
                "mod5": int(s % 5),
                "mod10": int(s % 10),
                "prompt": f"Output ONLY a number. {a}+{b}=",
                "target_token_id": int(tid),
            })
    return rows


examples = make_examples()


def get_last_positions(attention_mask):
    return attention_mask.sum(dim=1) - 1


def pred_token_to_int(pred_id):
    txt = tokenizer.decode([int(pred_id)]).strip()
    try:
        return int(txt)
    except Exception:
        return None


@torch.no_grad()
def get_baseline_correct_examples(max_examples=None):
    rows = examples if max_examples is None else examples[:max_examples]
    kept = []

    for start in tqdm(range(0, len(rows), BATCH_SIZE), desc="baseline filter"):
        batch = rows[start:start+BATCH_SIZE]
        enc = tokenizer(
            [x["prompt"] for x in batch],
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(model.device)

        last_pos = get_last_positions(enc["attention_mask"])
        batch_idx = torch.arange(len(batch), device=model.device)

        out = model(**enc, use_cache=False)
        pred_ids = out.logits[batch_idx, last_pos].argmax(dim=-1).detach().cpu().tolist()

        for ex, pred_id in zip(batch, pred_ids):
            if int(pred_id) == int(ex["target_token_id"]):
                kept.append(ex)

    return kept


def make_fourier_basis(sums, period):
    s = sums.astype(np.float64)
    theta = 2.0 * np.pi * s / float(period)
    return np.column_stack([np.cos(theta), np.sin(theta)])


def learn_plane_Q(stream, layer, period):
    x_path = os.path.join(MLP_DIR, f"{stream}_layer{layer:02d}_eq_correct_only.pt")
    m_path = os.path.join(MLP_DIR, f"metadata_layer{layer:02d}_correct_only.jsonl")

    X = torch.load(x_path, map_location="cpu").float().numpy()

    meta = []
    with open(m_path, "r", encoding="utf-8") as f:
        for line in f:
            meta.append(json.loads(line))

    sums = np.array([m["sum"] for m in meta], dtype=np.float64)
    Y = make_fourier_basis(sums, period)

    reg = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
    reg.fit(Y, X)

    W = reg.coef_  # [D, 2]
    Q, _ = np.linalg.qr(W)  # [D, 2], orthonormal

    return torch.tensor(Q[:, :2], dtype=torch.float16, device=model.device)


Q = learn_plane_Q(STREAM, LAYER, PERIOD)
print("Q:", Q.shape, "stream:", STREAM, "layer:", LAYER, "period:", PERIOD)


def get_modules(layer):
    mlp = model.gpt_neox.layers[layer].mlp
    return {
        "dense_h_to_4h": mlp.dense_h_to_4h,
        "dense_4h_to_h": mlp.dense_4h_to_h,
    }


def edit_phase_or_radius(x, Q, mode, period=5, delta=1, scale=1.0):
    """
    x: [B, D]
    Q: [D, 2] orthonormal plane directions
    mode:
      "rotate"  -> theta += 2*pi*delta/period
      "scale"   -> radius *= scale
      "scramble"-> random theta per sample, same radius
    """
    coords = x @ Q  # [B, 2]
    rest = x - coords @ Q.T

    cx = coords[:, 0]
    cy = coords[:, 1]

    r = torch.sqrt(cx * cx + cy * cy + 1e-12)
    theta = torch.atan2(cy, cx)

    if mode == "rotate":
        theta2 = theta + (2.0 * np.pi * delta / period)
        r2 = r

    elif mode == "scale":
        theta2 = theta
        r2 = r * scale

    elif mode == "scramble":
        theta2 = torch.rand_like(theta) * (2.0 * np.pi)
        r2 = r

    else:
        raise ValueError(mode)

    new_coords = torch.stack(
        [r2 * torch.cos(theta2), r2 * torch.sin(theta2)],
        dim=-1,
    )

    return rest + new_coords @ Q.T


def register_edit_hook(stream, layer, Q, mode, period=5, delta=1, scale=1.0):
    modules = get_modules(layer)
    dense_h_to_4h = modules["dense_h_to_4h"]
    dense_4h_to_h = modules["dense_4h_to_h"]

    if stream == "mlp_input":
        def pre_hook(module, inputs):
            x = inputs[0].clone()
            x[batch_idx_global, last_pos_global] = edit_phase_or_radius(
                x[batch_idx_global, last_pos_global],
                Q,
                mode=mode,
                period=period,
                delta=delta,
                scale=scale,
            )
            return (x,)
        return dense_h_to_4h.register_forward_pre_hook(pre_hook)

    elif stream == "preact":
        def hook(module, inputs, output):
            y = output.clone()
            y[batch_idx_global, last_pos_global] = edit_phase_or_radius(
                y[batch_idx_global, last_pos_global],
                Q,
                mode=mode,
                period=period,
                delta=delta,
                scale=scale,
            )
            return y
        return dense_h_to_4h.register_forward_hook(hook)

    elif stream == "postact":
        def pre_hook(module, inputs):
            x = inputs[0].clone()
            x[batch_idx_global, last_pos_global] = edit_phase_or_radius(
                x[batch_idx_global, last_pos_global],
                Q,
                mode=mode,
                period=period,
                delta=delta,
                scale=scale,
            )
            return (x,)
        return dense_4h_to_h.register_forward_pre_hook(pre_hook)

    elif stream == "mlp_out":
        def hook(module, inputs, output):
            y = output.clone()
            y[batch_idx_global, last_pos_global] = edit_phase_or_radius(
                y[batch_idx_global, last_pos_global],
                Q,
                mode=mode,
                period=period,
                delta=delta,
                scale=scale,
            )
            return y
        return dense_4h_to_h.register_forward_hook(hook)

    else:
        raise ValueError(stream)


@torch.no_grad()
def eval_edit(rows, mode=None, period=5, delta=1, scale=1.0):
    stats = {
        "n": 0,
        "exact": 0,
        "modT_original": 0,
        "modT_shifted": 0,
        "parseable": 0,
    }

    hook = None
    if mode is not None:
        hook = register_edit_hook(
            STREAM,
            LAYER,
            Q,
            mode=mode,
            period=period,
            delta=delta,
            scale=scale,
        )

    try:
        for start in tqdm(range(0, len(rows), BATCH_SIZE), desc=f"eval {mode or 'baseline'}"):
            batch = rows[start:start+BATCH_SIZE]
            enc = tokenizer(
                [x["prompt"] for x in batch],
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            ).to(model.device)

            global batch_idx_global, last_pos_global
            last_pos_global = get_last_positions(enc["attention_mask"])
            batch_idx_global = torch.arange(len(batch), device=model.device)

            out = model(**enc, use_cache=False)
            pred_ids = out.logits[batch_idx_global, last_pos_global].argmax(dim=-1).detach().cpu().tolist()

            for pred_id, ex in zip(pred_ids, batch):
                stats["n"] += 1

                if int(pred_id) == int(ex["target_token_id"]):
                    stats["exact"] += 1

                pred_num = pred_token_to_int(pred_id)
                if pred_num is None:
                    continue

                stats["parseable"] += 1

                true_residue = ex["sum"] % period
                shifted_residue = (true_residue + delta) % period

                stats["modT_original"] += int(pred_num % period == true_residue)
                stats["modT_shifted"] += int(pred_num % period == shifted_residue)

    finally:
        if hook is not None:
            hook.remove()

    n = max(stats["n"], 1)
    return {k: (v / n if k != "n" else v) for k, v in stats.items()}


baseline_correct = get_baseline_correct_examples(max_examples=None)
eval_rows = baseline_correct[:MAX_EVAL_EXAMPLES]

print("eval rows:", len(eval_rows), "baseline-correct total:", len(baseline_correct))

tests = [
    ("baseline", None, None),
    (f"rotate_T{PERIOD}_plus{DELTA}", "rotate", {"period": PERIOD, "delta": DELTA, "scale": 1.0}),
    (f"scale_T{PERIOD}_x0.5", "scale", {"period": PERIOD, "delta": DELTA, "scale": 0.5}),
    (f"scale_T{PERIOD}_x2.0", "scale", {"period": PERIOD, "delta": DELTA, "scale": 2.0}),
    (f"scramble_T{PERIOD}", "scramble", {"period": PERIOD, "delta": DELTA, "scale": 1.0}),
]

print("\nRESULTS")
print("-" * 100)
print(f"{'test':<24} {'n':>5} {'exact':>8} {'orig_modT':>10} {'shift_modT':>10} {'parse':>8}")
print("-" * 100)

for name, mode, kwargs in tests:
    if mode is None:
        res = eval_edit(eval_rows, mode=None, period=PERIOD, delta=DELTA)
    else:
        res = eval_edit(eval_rows, mode=mode, **kwargs)

    print(
        f"{name:<24} {res['n']:>5d} "
        f"{res['exact']:>8.4f} "
        f"{res['modT_original']:>10.4f} "
        f"{res['modT_shifted']:>10.4f} "
        f"{res['parseable']:>8.4f}"
    )

## Numbered main residual pipeline

Run this section for the Pythia residual-stream experiments. Tests are numbered explicitly below.

Numbering in the main pipeline:

- **TEST 1**: setup + persistent low-period residual ablation sweep
- **TEST 2**: period-wise ablation + **TEST 2B** decade/same-units analysis
- **TEST 3**: wrong-frequency orthogonalized controls
- **TEST 4**: empirical spectrum + Hfull controls
- **TEST 5**: frequency → residue matrix
- **TEST 6**: low-period pin/replacement steering experiment
- **TEST 7**: T100 / LINEAR magnitude steering



In [ ]:
# ============================================================
# REQUIRED SETUP FOR MAIN ABLATION / TEST 1-7
# Run this if you skipped the optional earlier cells.
# ============================================================

import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch

from tqdm import tqdm
from sklearn.linear_model import Ridge
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")

# -----------------------------
# Config
# -----------------------------

MODEL_NAME = "EleutherAI/pythia-6.9b"

A_MAX = 99
B_MAX = 99
BATCH_SIZE = 32
RIDGE_ALPHA = 1.0
SEED = 42

# Set to None for full baseline-correct eval.
# Set to 1000 for smoke test.
MAX_EVAL_EXAMPLES = None

SAVE_RESULTS = True
OUT_DIR = "/content/pythia_crt_ablation_updated"

# -----------------------------
# Helpers
# -----------------------------

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

set_seed(SEED)

# -----------------------------
# Load tokenizer/model
# -----------------------------

print_section("LOAD TOKENIZER")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

print("pad_token:", repr(tokenizer.pad_token), tokenizer.pad_token_id)
print("eos_token:", repr(tokenizer.eos_token), tokenizer.eos_token_id)
print("padding_side:", tokenizer.padding_side)

print_section("LOAD MODEL")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
).eval()

model.config.pad_token_id = tokenizer.pad_token_id

print("model:", MODEL_NAME)
print("device:", model.device)
print("dtype:", next(model.parameters()).dtype)
print("num layers:", model.config.num_hidden_layers)

# -----------------------------
# Dataset
# -----------------------------

def answer_token_id(n):
    ids = tokenizer(str(int(n)), add_special_tokens=False)["input_ids"]
    if len(ids) != 1:
        return None
    return int(ids[0])

def make_prompt(a, b):
    return f"Output ONLY a number. {a}+{b}="

def make_examples():
    rows = []
    bad = []

    for a in range(A_MAX + 1):
        for b in range(B_MAX + 1):
            s = a + b
            tid = answer_token_id(s)

            if tid is None:
                bad.append((a, b, s))
                continue

            rows.append({
                "a": int(a),
                "b": int(b),
                "sum": int(s),
                "mod2": int(s % 2),
                "mod3": int(s % 3),
                "mod5": int(s % 5),
                "mod7": int(s % 7),
                "mod10": int(s % 10),
                "prompt": make_prompt(a, b),
                "target_token_id": int(tid),
            })

    return rows, bad

def get_last_positions(attention_mask):
    return attention_mask.sum(dim=1) - 1

def pred_token_to_int(pred_id):
    txt = tokenizer.decode([int(pred_id)]).strip()
    try:
        return int(txt)
    except Exception:
        return None

examples, bad_answer_examples = make_examples()

print_section("DATASET")
print("examples:", len(examples))
print("bad answer examples:", len(bad_answer_examples))
print("first example:", examples[0])
print("last example:", examples[-1])

# -----------------------------
# Baseline-correct filter
# -----------------------------

@torch.no_grad()
def get_baseline_correct_examples(max_examples=None):
    rows = examples if max_examples is None else examples[:max_examples]
    kept = []
    pred_rows = []

    for start in tqdm(range(0, len(rows), BATCH_SIZE), desc="baseline filter"):
        batch = rows[start:start + BATCH_SIZE]

        enc = tokenizer(
            [x["prompt"] for x in batch],
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(model.device)

        last_pos = get_last_positions(enc["attention_mask"])
        batch_idx = torch.arange(len(batch), device=model.device)

        out = model(**enc, use_cache=False)
        pred_ids = out.logits[batch_idx, last_pos].argmax(dim=-1).detach().cpu().tolist()

        for ex, pred_id in zip(batch, pred_ids):
            pred_num = pred_token_to_int(pred_id)
            correct = int(pred_id) == int(ex["target_token_id"])

            rec = {
                **ex,
                "pred_token_id": int(pred_id),
                "pred": pred_num,
                "correct": correct,
            }
            pred_rows.append(rec)

            if correct:
                kept.append(rec)

    return kept, pred_rows

print_section("BASELINE-CORRECT FILTER")

baseline_correct_examples, baseline_pred_rows = get_baseline_correct_examples(max_examples=None)

print("baseline correct:", len(baseline_correct_examples), "/", len(examples))
print("baseline exact acc:", len(baseline_correct_examples) / len(examples))

if MAX_EVAL_EXAMPLES is None:
    eval_examples = baseline_correct_examples
else:
    eval_examples = baseline_correct_examples[:MAX_EVAL_EXAMPLES]

print("eval examples:", len(eval_examples))

# Optional natural error quick summary
wrong_rows = [r for r in baseline_pred_rows if not r["correct"]]
parse_wrong = [r for r in wrong_rows if r["pred"] is not None]

if len(parse_wrong):
    diffs = np.array([r["pred"] - r["sum"] for r in parse_wrong], dtype=int)
    vals, counts = np.unique(diffs, return_counts=True)

    print("\nTop baseline error shifts among parseable wrong:")
    for i in np.argsort(-counts)[:10]:
        print(f"  {int(vals[i]):+d}: {int(counts[i])} ({counts[i] / len(diffs) * 100:.1f}%)")

    print("\nResidue preserved among parseable wrong:")
    for k in [2, 3, 5, 7, 10]:
        keep = np.mean([r["pred"] % k == r["sum"] % k for r in parse_wrong])
        print(f"  mod{k:<2}: {keep * 100:.1f}%")

In [ ]:
# ============================================================
# TEST 0: BASELINE PREDICTIONS / NATURAL ERROR TABLE
#
# Purpose:
#   Save the model's unablated predictions on the full addition set.
#
# Produces:
#   baseline_predictions.csv
#
# This file is used by the human-readable dashboard to support:
#   - baseline exact accuracy
#   - natural error shifts
#   - whether natural wrong answers preserve residues / units
#   - ±10 and multiple-of-10 error structure
#
# Place this cell after:
#   baseline_correct_examples, baseline_pred_rows = get_baseline_correct_examples(...)
#
# and before TEST 1.
# ============================================================

import os
import numpy as np
import pandas as pd

print("\n" + "=" * 100)
print("TEST 0: BASELINE PREDICTIONS / NATURAL ERRORS")
print("=" * 100)

# Make sure OUT_DIR exists even if save_df has not been defined yet.
OUT_DIR = globals().get("OUT_DIR", "/content/pythia_crt_ablation_updated")
SAVE_RESULTS = globals().get("SAVE_RESULTS", True)

if "baseline_pred_rows" not in globals():
    raise RuntimeError(
        "baseline_pred_rows is missing. Run the baseline-correct filter cell first:\n"
        "baseline_correct_examples, baseline_pred_rows = get_baseline_correct_examples(max_examples=None)"
    )

baseline_predictions_df = pd.DataFrame(baseline_pred_rows).copy()

required_cols = {"a", "b", "sum", "pred", "correct"}
missing_cols = required_cols - set(baseline_predictions_df.columns)
if missing_cols:
    raise RuntimeError(f"baseline_pred_rows is missing required columns: {sorted(missing_cols)}")

# ------------------------------------------------------------
# Normalize dtypes safely
# ------------------------------------------------------------

baseline_predictions_df["correct"] = baseline_predictions_df["correct"].astype(bool)
baseline_predictions_df["parseable"] = baseline_predictions_df["pred"].notna()

# Numeric pred column with NaN for unparseable outputs.
baseline_predictions_df["pred_num"] = pd.to_numeric(
    baseline_predictions_df["pred"],
    errors="coerce",
)

baseline_predictions_df["diff"] = (
    baseline_predictions_df["pred_num"] - baseline_predictions_df["sum"]
)

baseline_predictions_df["abs_err"] = baseline_predictions_df["diff"].abs()

# ------------------------------------------------------------
# Residue preservation columns
# ------------------------------------------------------------

for k in [2, 3, 5, 7, 10]:
    baseline_predictions_df[f"same_mod{k}"] = (
        baseline_predictions_df["parseable"]
        & ((baseline_predictions_df["pred_num"] % k) == (baseline_predictions_df["sum"] % k))
    )

baseline_predictions_df["same_units"] = baseline_predictions_df["same_mod10"]

baseline_predictions_df["same_decade"] = (
    baseline_predictions_df["parseable"]
    & ((baseline_predictions_df["pred_num"] // 10) == (baseline_predictions_df["sum"] // 10))
)

wrong_mask = baseline_predictions_df["parseable"] & (~baseline_predictions_df["correct"])

baseline_predictions_df["wrong_parseable"] = wrong_mask

baseline_predictions_df["multiple_of_10_error"] = (
    wrong_mask
    & ((baseline_predictions_df["diff"].abs() % 10) == 0)
)

baseline_predictions_df["pm10_error"] = (
    wrong_mask
    & (baseline_predictions_df["diff"].isin([-10, 10]))
)

# Helpful human-readable error type.
baseline_predictions_df["error_type"] = "correct"
baseline_predictions_df.loc[~baseline_predictions_df["parseable"], "error_type"] = "unparseable"
baseline_predictions_df.loc[wrong_mask & baseline_predictions_df["pm10_error"], "error_type"] = "±10_error"
baseline_predictions_df.loc[
    wrong_mask
    & (~baseline_predictions_df["pm10_error"])
    & baseline_predictions_df["multiple_of_10_error"],
    "error_type",
] = "multiple_of_10_error"
baseline_predictions_df.loc[
    wrong_mask
    & (~baseline_predictions_df["multiple_of_10_error"]),
    "error_type",
] = "other_parseable_wrong"

# ------------------------------------------------------------
# Print compact summary
# ------------------------------------------------------------

n_total = len(baseline_predictions_df)
n_correct = int(baseline_predictions_df["correct"].sum())
n_parseable = int(baseline_predictions_df["parseable"].sum())
n_wrong_parseable = int(wrong_mask.sum())

print(f"n_total:           {n_total}")
print(f"baseline correct:  {n_correct} / {n_total} = {n_correct / max(n_total, 1):.4f}")
print(f"parseable:         {n_parseable} / {n_total} = {n_parseable / max(n_total, 1):.4f}")
print(f"parseable wrong:   {n_wrong_parseable}")

if n_wrong_parseable > 0:
    wrong_df = baseline_predictions_df.loc[wrong_mask].copy()

    print("\nResidue preserved among parseable wrong:")
    for k in [2, 3, 5, 7, 10]:
        val = wrong_df[f"same_mod{k}"].mean()
        print(f"  mod{k:<2}: {val * 100:.1f}%")

    print("\nNatural error structure among parseable wrong:")
    print(f"  multiple_of_10_error: {wrong_df['multiple_of_10_error'].mean() * 100:.1f}%")
    print(f"  ±10_error:            {wrong_df['pm10_error'].mean() * 100:.1f}%")
    print(f"  same_units:           {wrong_df['same_units'].mean() * 100:.1f}%")
    print(f"  same_decade:          {wrong_df['same_decade'].mean() * 100:.1f}%")
    print(f"  median_abs_err:       {wrong_df['abs_err'].median():.1f}")

    print("\nTop baseline error shifts among parseable wrong:")
    shift_counts = wrong_df["diff"].value_counts().head(15)
    for diff, count in shift_counts.items():
        print(f"  {int(diff):+d}: {int(count)} ({count / n_wrong_parseable * 100:.1f}%)")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

if SAVE_RESULTS:
    os.makedirs(OUT_DIR, exist_ok=True)
    baseline_path = os.path.join(OUT_DIR, "baseline_predictions.csv")
    baseline_predictions_df.to_csv(baseline_path, index=False)
    print("\nsaved:", baseline_path)
else:
    print("\nSAVE_RESULTS=False, not saving baseline_predictions.csv")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.linear_model import Ridge

# ============================================================
# TEST 1: SETUP + PERSISTENT T2/T5/T10 RESIDUAL ABLATION SWEEP
#
# Adds GPT-J-parity tests to Pythia:
#   T100, LINEAR, T100_LINEAR, T2T5T10T100
#   decade / same-units error analysis
#   richer residue matrix
# ============================================================

N_LAYERS = model.config.num_hidden_layers
PERIODS = [2, 5, 10]
INCL_LIN = False
LAYER_MIN = 10
L0 = 17
FIT_N = 4000
RANDOM_CONTROL_SEEDS = [0, 1, 2, 3, 4]
SAVE_RESULTS = True
OUT_DIR = globals().get("OUT_DIR", "/content/pythia_crt_ablation_updated")


def save_df(df, name):
    if not SAVE_RESULTS:
        return
    os.makedirs(OUT_DIR, exist_ok=True)
    path = os.path.join(OUT_DIR, name)
    df.to_csv(path, index=False)
    print("saved:", path)


def make_fourier_basis(sums, periods, include_linear=False):
    s = np.asarray(sums, dtype=np.float64)
    cols = []

    if include_linear:
        z = (s - s.mean()) / (s.std() + 1e-12)
        cols.append(z)

    for T in periods:
        theta = 2.0 * np.pi * s / float(T)
        cols.append(np.cos(theta))
        cols.append(np.sin(theta))

    return np.column_stack(cols)


# ---- 1) one pass: collect residual h^l at the '=' token, per layer, to FIT Q_l ----
def collect_resids(rows, layers):
    store = {l: [] for l in layers}
    S = []
    buf = {}

    def make_hook(L):
        def hook(module, inputs, output):
            hs = output[0] if isinstance(output, tuple) else output
            buf[L] = hs.detach()
        return hook

    handles = [model.gpt_neox.layers[l].register_forward_hook(make_hook(l)) for l in layers]

    try:
        for st in tqdm(range(0, len(rows), BATCH_SIZE), desc="collect residuals"):
            b = rows[st:st + BATCH_SIZE]
            enc = tokenizer(
                [x["prompt"] for x in b],
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            ).to(model.device)

            lp = get_last_positions(enc["attention_mask"])
            bi = torch.arange(len(b), device=model.device)

            buf.clear()
            with torch.no_grad():
                model(**enc, use_cache=False)

            for l in layers:
                if l not in buf:
                    raise RuntimeError(f"Missing hook output for layer {l}")
                store[l].append(buf[l][bi, lp].float().cpu().numpy())

            S += [x["sum"] for x in b]

    finally:
        for h in handles:
            h.remove()

    return {l: np.concatenate(store[l], axis=0) for l in layers}, np.array(S, dtype=np.float64)


def fit_Q(X, s, periods=PERIODS, incl=None, include_linear=None):
    # Backward-compatible: older cells use incl=..., newer code uses include_linear=...
    if include_linear is None:
        include_linear = INCL_LIN if incl is None else incl

    Y = make_fourier_basis(s, periods, include_linear=include_linear)
    W = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True).fit(Y, X).coef_
    Q, _ = np.linalg.qr(W)
    return torch.tensor(Q, dtype=torch.float16, device=model.device)


def fit_lin_Q(X, s):
    s = np.asarray(s, dtype=np.float64)
    z = ((s - s.mean()) / (s.std() + 1e-12)).reshape(-1, 1)
    W = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True).fit(z, X).coef_
    Q, _ = np.linalg.qr(W)
    return torch.tensor(Q, dtype=torch.float16, device=model.device)


def rand_Q(D, k, seed=0):
    g = torch.Generator(device="cpu").manual_seed(seed)
    A = torch.randn(D, k, generator=g, dtype=torch.float32)
    Q, _ = torch.linalg.qr(A)
    return Q.to(dtype=torch.float16, device=model.device)


# ---- 2) PERSISTENT residual ablation at '=' token for all specified layers ----
_lp_g = None
_bi_g = None


def make_resid_hook(Q):
    def h(m, i, o):
        if isinstance(o, tuple):
            hs = o[0].clone()
            rest = tuple(o[1:])
        else:
            hs = o.clone()
            rest = None

        v = hs[_bi_g, _lp_g]
        hs[_bi_g, _lp_g] = v - (v @ Q) @ Q.T

        return hs if rest is None else (hs,) + rest
    return h


@torch.no_grad()
def eval_persistent(rows, Q_by_layer, ks=(2, 3, 5, 7, 10), label="", show_progress=True, return_records=False):
    global _lp_g, _bi_g

    handles = [
        model.gpt_neox.layers[l].register_forward_hook(make_resid_hook(Q))
        for l, Q in Q_by_layer.items()
    ]

    n = exact = parseable = 0
    hit = {k: 0 for k in ks}
    diffs_all = []
    diffs_wrong = []
    pred_records = []

    iterator = range(0, len(rows), BATCH_SIZE)
    if show_progress:
        iterator = tqdm(iterator, desc=f"eval persistent {label}")

    try:
        for st in iterator:
            b = rows[st:st + BATCH_SIZE]
            enc = tokenizer(
                [x["prompt"] for x in b],
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            ).to(model.device)

            _lp_g = get_last_positions(enc["attention_mask"])
            _bi_g = torch.arange(len(b), device=model.device)

            logits = model(**enc, use_cache=False).logits[_bi_g, _lp_g]
            pred_ids = logits.argmax(-1).detach().cpu().tolist()

            for pid, ex in zip(pred_ids, b):
                n += 1
                is_exact = int(pid) == int(ex["target_token_id"])
                exact += int(is_exact)

                pn = pred_token_to_int(pid)

                if return_records:
                    pred_records.append({
                        "a": ex["a"],
                        "b": ex["b"],
                        "sum": ex["sum"],
                        "pred_token_id": int(pid),
                        "pred": pn,
                        "exact": int(is_exact),
                    })

                if pn is None:
                    continue

                parseable += 1
                for k in ks:
                    hit[k] += int(pn % k == int(ex["sum"]) % k)

                diff = pn - int(ex["sum"])
                diffs_all.append(diff)
                if not is_exact:
                    diffs_wrong.append(diff)

    finally:
        for h in handles:
            h.remove()

    n_safe = max(n, 1)
    diffs_all = np.array(diffs_all)
    diffs_wrong = np.array(diffs_wrong)

    out = {
        "n": n,
        "exact": exact / n_safe,
        "parseable": parseable / n_safe,
        "median_abs_err_all": float(np.median(np.abs(diffs_all))) if len(diffs_all) else None,
        "median_abs_err_wrong": float(np.median(np.abs(diffs_wrong))) if len(diffs_wrong) else None,
        "frac_wrong_units": float(np.mean(np.abs(diffs_all) % 10 != 0)) if len(diffs_all) else None,
    }
    for k in ks:
        out[f"mod{k}"] = hit[k] / n_safe

    if return_records:
        return out, pred_records
    return out


# Older downstream cells call eval_persistent_err; keep it compatible.
def eval_persistent_err(rows, Q_by_layer):
    return eval_persistent(rows, Q_by_layer, ks=(2, 3, 5, 7, 10), show_progress=False)


# ---- TEST 1 RUN: sweep L_start, low-period helix vs random control ----
print("\n" + "=" * 100)
print("TEST 1: PERSISTENT LOW-PERIOD RESIDUAL ABLATION SWEEP")
print("=" * 100)

layers = list(range(LAYER_MIN, N_LAYERS))
print("layers:", layers)
print("main ablation start L0:", L0)
print("fit examples:", min(FIT_N, len(baseline_correct_examples)))

Xres, Sres = collect_resids(baseline_correct_examples[:FIT_N], layers)
Qh = {l: fit_Q(Xres[l], Sres, periods=PERIODS, include_linear=INCL_LIN) for l in layers}
Qr = {l: rand_Q(Xres[l].shape[1], Qh[l].shape[1], seed=l) for l in layers}

sweep_rows = []
print(f"{'L_start':>7} {'helix_exact':>12} {'rand_exact':>11} {'helix_mod10':>12} {'rand_mod10':>11} {'n_layers':>8}")
for L in range(N_LAYERS - 1, LAYER_MIN - 1, -3):
    sub_h = {l: Qh[l] for l in layers if l >= L}
    sub_r = {l: Qr[l] for l in layers if l >= L}
    rh = eval_persistent(eval_examples, sub_h, label=f"helix L>={L}")
    rr = eval_persistent(eval_examples, sub_r, label=f"random L>={L}")
    print(f"{L:>7} {rh['exact']:>12.4f} {rr['exact']:>11.4f} {rh['mod10']:>12.4f} {rr['mod10']:>11.4f} {len(sub_h):>8}")
    sweep_rows.append({"condition": "helix", "L_start": L, "n_layers": len(sub_h), **rh})
    sweep_rows.append({"condition": "random_seed0", "L_start": L, "n_layers": len(sub_r), **rr})

sweep_df = pd.DataFrame(sweep_rows)
save_df(sweep_df, "test1_persistent_sweep.csv")

In [ ]:
# ============================================================
# TEST 2: PERIOD-WISE PERSISTENT ABLATION + DECADE/SAME-UNITS ANALYSIS
# GPT-J parity additions:
#   T100, LINEAR, T100_LINEAR, T2T5T10T100
#   plus decade / same-units analysis
# ============================================================

L0 = 17

SETS = [
    ("T2", [2]),
    ("T5", [5]),
    ("T10", [10]),
    ("T100", [100]),
    ("LINEAR", []),
    ("T100_LINEAR", [100]),
    ("T3_ctrl", [3]),
    ("T7_ctrl", [7]),
    ("T2T5T10", [2, 5, 10]),
    ("T2T5T10T100", [2, 5, 10, 100]),
]

periodwise_rows = []
periodwise_pred_records = {}

print("\n" + "=" * 100)
print("TEST 2: PERIOD-WISE PERSISTENT ABLATION")
print("=" * 100)
print(
    f"{'subspace':14} {'exact':>7} {'mod2':>6} {'mod3':>6} {'mod5':>6} "
    f"{'mod7':>6} {'mod10':>6} {'med|err|wrong':>14} {'wrong_units':>12} {'parse':>7}"
)
print("-" * 120)

for name, P in SETS:
    if name == "LINEAR":
        Q = {l: fit_lin_Q(Xres[l], Sres) for l in layers if l >= L0}
    elif name == "T100_LINEAR":
        Q = {l: fit_Q(Xres[l], Sres, periods=P, include_linear=True) for l in layers if l >= L0}
    else:
        Q = {l: fit_Q(Xres[l], Sres, periods=P, include_linear=False) for l in layers if l >= L0}

    r, preds = eval_persistent(eval_examples, Q, label=name, return_records=True)
    periodwise_pred_records[name] = preds

    periodwise_rows.append({
        "subspace": name,
        "periods": str(P),
        "include_linear": bool(name in ["LINEAR", "T100_LINEAR"]),
        "random_seed": None,
        **r,
    })

    print(
        f"{name:14} {r['exact']:>7.3f} {r['mod2']:>6.3f} {r['mod3']:>6.3f} "
        f"{r['mod5']:>6.3f} {r['mod7']:>6.3f} {r['mod10']:>6.3f} "
        f"{(r['median_abs_err_wrong'] or 0):>14.1f} {(r['frac_wrong_units'] or 0):>12.3f} "
        f"{r['parseable']:>7.3f}"
    )

for seed in RANDOM_CONTROL_SEEDS:
    Q = {l: rand_Q(Xres[l].shape[1], 6, seed=seed * 1000 + l) for l in layers if l >= L0}
    r = eval_persistent(eval_examples, Q, label=f"RANDOM_rank6_seed{seed}")

    periodwise_rows.append({
        "subspace": "RANDOM_rank6",
        "periods": None,
        "include_linear": False,
        "random_seed": seed,
        **r,
    })

    print(
        f"{'RANDOM'+str(seed):14} {r['exact']:>7.3f} {r['mod2']:>6.3f} {r['mod3']:>6.3f} "
        f"{r['mod5']:>6.3f} {r['mod7']:>6.3f} {r['mod10']:>6.3f} "
        f"{(r['median_abs_err_wrong'] or 0):>14.1f} {(r['frac_wrong_units'] or 0):>12.3f} "
        f"{r['parseable']:>7.3f}"
    )

periodwise_df = pd.DataFrame(periodwise_rows)
save_df(periodwise_df, "test2_periodwise_ablation.csv")

rand6 = periodwise_df[periodwise_df["subspace"] == "RANDOM_rank6"]
if len(rand6):
    print("\nRANDOM rank-6 summary:")
    print(rand6[["exact", "mod2", "mod3", "mod5", "mod7", "mod10"]].agg(["mean", "std"]))


# ============================================================
# TEST 2B: DECADE / SAME-UNITS ERROR ANALYSIS
# ============================================================

print("\n" + "=" * 100)
print("TEST 2B: DECADE / SAME-UNITS ERROR ANALYSIS")
print("=" * 100)


def summarize_pred_records(preds, label):
    rows = [r for r in preds if r["pred"] is not None]
    if not rows:
        return {"label": label, "parseable_n": 0}

    diffs = np.array([r["pred"] - r["sum"] for r in rows], dtype=int)
    exact = np.array([r["exact"] for r in rows], dtype=int)
    same_units = np.array([r["pred"] % 10 == r["sum"] % 10 for r in rows], dtype=bool)
    same_decade = np.array([r["pred"] // 10 == r["sum"] // 10 for r in rows], dtype=bool)
    wrong_mask = exact == 0

    out = {
        "label": label,
        "parseable_n": len(rows),
        "exact_parseable": float(exact.mean()),
        "same_units_all": float(same_units.mean()),
        "same_decade_all": float(same_decade.mean()),
        "median_abs_err_all": float(np.median(np.abs(diffs))),
    }

    if wrong_mask.sum():
        out.update({
            "wrong_n": int(wrong_mask.sum()),
            "same_units_wrong": float(same_units[wrong_mask].mean()),
            "same_decade_wrong": float(same_decade[wrong_mask].mean()),
            "multiple_of_10_wrong": float(np.mean(np.abs(diffs[wrong_mask]) % 10 == 0)),
            "pm10_wrong": float(np.mean(np.isin(diffs[wrong_mask], [-10, 10]))),
            "median_abs_err_wrong": float(np.median(np.abs(diffs[wrong_mask]))),
        })

    return out


decade_rows = []
for label in ["T2T5T10", "T5", "T100", "LINEAR", "T100_LINEAR", "T2T5T10T100"]:
    if label in periodwise_pred_records:
        decade_rows.append(summarize_pred_records(periodwise_pred_records[label], label))

decade_df = pd.DataFrame(decade_rows)
print(decade_df)
save_df(decade_df, "extra_decade_same_units_analysis.csv")

In [ ]:
# ============================================================
# TEST 3: WRONG-FREQUENCY OVERLAP / ORTHOGONALIZED CONTROLS
#
# Purpose:
#   Test whether raw T3/T7 ablation damage is mostly caused by
#   overlap/leakage with the real addition helix span.
#
# Main reviewer-facing control:
#   Orthogonalize T3/T7 only against Hspan:
#
#       Hspan = span(T2, T5, T10, T100, LINEAR)
#
#   This is intentionally tighter than Hfull. It does NOT include
#   extra empirical/PCA magnitude bases, so recovery after
#   orthogonalization is harder to dismiss as over-projection.
#
# Requires existing globals:
#   model, Xres, Sres, layers, L0
#   RIDGE_ALPHA
#   fit_Q, rand_Q
#   eval_examples, eval_persistent_err
#   save_df
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge

print("\n" + "=" * 100)
print("TEST 3: WRONG-FREQUENCY OVERLAP / ORTHOGONALIZED CONTROL — Hspan")
print("=" * 100)


# ------------------------------------------------------------
# Required-global checks
# ------------------------------------------------------------

_required = [
    "model",
    "Xres",
    "Sres",
    "layers",
    "L0",
    "RIDGE_ALPHA",
    "fit_Q",
    "rand_Q",
    "eval_examples",
    "eval_persistent_err",
    "save_df",
]

_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(f"Missing required globals for TEST 3: {_missing}")


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def orthobasis(Qs):
    """
    Concatenate basis matrices and return an orthonormal basis
    for their joint span.
    """
    if len(Qs) == 0:
        raise ValueError("orthobasis got an empty list.")

    M = torch.cat([Q.float() for Q in Qs], dim=1)

    if M.ndim != 2:
        raise ValueError(f"Expected 2D matrix after concat, got shape {tuple(M.shape)}")

    Q, _ = torch.linalg.qr(M, mode="reduced")
    return Q.to(dtype=torch.float16, device=model.device)


def fit_lin_Q_test3(X, s):
    """
    Fit a 1D linear/magnitude direction as a function of the target sum.
    Returns an orthonormal [D, 1] basis.
    """
    s = np.asarray(s, dtype=np.float64)
    z = ((s - s.mean()) / (s.std() + 1e-12)).reshape(-1, 1)

    reg = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
    reg.fit(z, X)

    W = reg.coef_  # [D, 1]
    Q, _ = np.linalg.qr(W)

    return torch.tensor(Q, dtype=torch.float16, device=model.device)


def princ_cos(A, B):
    """
    Principal-angle cosines between two orthonormal bases A and B.
    A: [D, k1], B: [D, k2]
    """
    return torch.linalg.svdvals(A.float().T @ B.float())


def orth_to(Qw, H, eps=1e-6):
    """
    Remove the component of Qw inside span(H), then re-orthonormalize.

    Returns:
        Q_orth: [D, rank(Qw)] basis after projection away from H
        norms:  column norms of the projected residual before QR

    If norms are near zero, the wrong-frequency basis was almost entirely
    inside Hspan.
    """
    Qw_f = Qw.float()
    H_f = H.float()

    R = Qw_f - H_f @ (H_f.T @ Qw_f)
    norms = torch.linalg.norm(R, dim=0)

    if torch.any(norms < eps):
        print(
            "WARNING: Some projected wrong-frequency directions have tiny residual norm:",
            norms.detach().cpu().numpy(),
        )

    Q, _ = torch.linalg.qr(R, mode="reduced")
    Q = Q[:, : Qw.shape[1]]

    return Q.to(dtype=torch.float16, device=model.device), norms.detach().cpu().numpy()


def metric_get(r, key):
    return r[key] if key in r else None


# ------------------------------------------------------------
# Build tight real helix span Hspan
# ------------------------------------------------------------

REAL_HSPAN_PERIODS = [2, 5, 10, 100]

Hspan = {}

for l in layers:
    if l < L0:
        continue

    basis_parts = [fit_Q(Xres[l], Sres, periods=[T]) for T in REAL_HSPAN_PERIODS]
    basis_parts.append(fit_lin_Q_test3(Xres[l], Sres))

    Hspan[l] = orthobasis(basis_parts)

if len(Hspan) == 0:
    raise RuntimeError(
        f"Hspan is empty. Check layers={layers} and L0={L0}."
    )

print("Hspan layers:", sorted(Hspan.keys()))
print("Hspan periods:", REAL_HSPAN_PERIODS, "+ LINEAR")
print("Hspan rank @ L0:", Hspan[L0].shape[1])


# ------------------------------------------------------------
# Random rank-2 anchor
# ------------------------------------------------------------

Qr2 = {
    l: rand_Q(Xres[l].shape[1], 2, seed=10_000 + int(l))
    for l in Hspan
}

r_rand2 = eval_persistent_err(eval_examples, Qr2)

print(
    f"RANDOM rank-2 anchor: "
    f"exact={r_rand2['exact']:.3f} "
    f"mod10={r_rand2['mod10']:.3f} "
    f"parseable={r_rand2.get('parseable', float('nan')):.3f}"
)


# ------------------------------------------------------------
# Raw vs Hspan-orthogonalized T3/T7 controls
# ------------------------------------------------------------

wrong_rows = []

print(
    f"\n{'freq':<6} {'cos@L0':<18} {'resid_norm@L0':<24} "
    f"{'raw_exact':>10} {'orth_exact':>11} "
    f"{'raw_m10':>9} {'orth_m10':>10} "
    f"{'Δexact':>9} {'Δm10':>9}"
)
print("-" * 120)

for T in [3, 7]:
    # Raw wrong-frequency basis, per layer.
    Qw = {
        l: fit_Q(Xres[l], Sres, periods=[T], include_linear=False)
        for l in Hspan
    }

    # Orthogonalized wrong-frequency basis, per layer.
    Qw_orth = {}
    norms_by_layer = {}

    for l in Hspan:
        Q_orth, norms = orth_to(Qw[l], Hspan[l])
        Qw_orth[l] = Q_orth
        norms_by_layer[l] = norms

    # Diagnostics at L0.
    cos_l0 = princ_cos(Qw[L0], Hspan[L0]).detach().cpu().numpy()
    norms_l0 = norms_by_layer[L0]

    # Evaluate damage.
    r_raw = eval_persistent_err(eval_examples, Qw)
    r_orth = eval_persistent_err(eval_examples, Qw_orth)

    exact_recovery = r_orth["exact"] - r_raw["exact"]
    mod10_recovery = r_orth["mod10"] - r_raw["mod10"]

    print(
        f"T{T:<5} "
        f"{[round(float(c), 3) for c in cos_l0]!s:<18} "
        f"{[round(float(n), 3) for n in norms_l0]!s:<24} "
        f"{r_raw['exact']:>10.3f} {r_orth['exact']:>11.3f} "
        f"{r_raw['mod10']:>9.3f} {r_orth['mod10']:>10.3f} "
        f"{exact_recovery:>9.3f} {mod10_recovery:>9.3f}"
    )

    wrong_rows.append({
        "control_set": "Hspan",
        "freq": f"T{T}",
        "period": T,
        "condition": "raw_vs_orth_to_Hspan",

        # Main dashboard columns.
        "raw_exact": r_raw["exact"],
        "orth_exact": r_orth["exact"],
        "raw_mod10": r_raw["mod10"],
        "orth_mod10": r_orth["mod10"],
        "exact_recovery": exact_recovery,
        "mod10_recovery": mod10_recovery,

        # Random anchor repeated for convenience.
        "random_rank2_exact": r_rand2["exact"],
        "random_rank2_mod10": r_rand2["mod10"],

        # Diagnostics.
        "cos_L0": str([round(float(c), 6) for c in cos_l0]),
        "resid_norm_L0": str([round(float(n), 6) for n in norms_l0]),
        "Hspan_rank_L0": int(Hspan[L0].shape[1]),

        # Compact secondary metrics.
        "raw_parseable": metric_get(r_raw, "parseable"),
        "orth_parseable": metric_get(r_orth, "parseable"),
        "raw_mod2": metric_get(r_raw, "mod2"),
        "orth_mod2": metric_get(r_orth, "mod2"),
        "raw_mod3": metric_get(r_raw, "mod3"),
        "orth_mod3": metric_get(r_orth, "mod3"),
        "raw_mod5": metric_get(r_raw, "mod5"),
        "orth_mod5": metric_get(r_orth, "mod5"),
        "raw_mod7": metric_get(r_raw, "mod7"),
        "orth_mod7": metric_get(r_orth, "mod7"),
        "raw_median_abs_err_wrong": metric_get(r_raw, "median_abs_err_wrong"),
        "orth_median_abs_err_wrong": metric_get(r_orth, "median_abs_err_wrong"),
        "raw_frac_wrong_units": metric_get(r_raw, "frac_wrong_units"),
        "orth_frac_wrong_units": metric_get(r_orth, "frac_wrong_units"),

        # Full metrics, namespaced for later dashboard use.
        **{f"raw_{k}": v for k, v in r_raw.items()},
        **{f"orth_{k}": v for k, v in r_orth.items()},
    })


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

wrong_df = pd.DataFrame(wrong_rows)

if len(wrong_df) != 2:
    raise RuntimeError(f"Expected exactly 2 Test 3 rows for T3/T7, got {len(wrong_df)}")

required_cols = [
    "control_set",
    "freq",
    "raw_exact",
    "orth_exact",
    "raw_mod10",
    "orth_mod10",
    "exact_recovery",
    "mod10_recovery",
]

missing_cols = [c for c in required_cols if c not in wrong_df.columns]
if missing_cols:
    raise RuntimeError(f"TEST 3 output missing required columns: {missing_cols}")

save_df(wrong_df, "test3_wrong_frequency_orthogonalized.csv")

print("\nSaved non-empty Test 3 CSV:")
print(wrong_df[required_cols + ["cos_L0", "resid_norm_L0", "random_rank2_exact", "random_rank2_mod10"]].to_string(index=False))

print("\nTEST 3 complete.")

In [ ]:
# ============================================================
# TEST 4: EMPIRICAL SPECTRUM + Hfull WRONG-FREQUENCY CONTROLS
# ============================================================

print("\n" + "=" * 100)
print("TEST 4: EMPIRICAL SPECTRUM + Hfull WRONG-FREQUENCY CONTROLS")
print("=" * 100)

# Identify the model's ACTUAL outlier periods (justifies the span; should be ~2,2.5,5,10 — not 3,7)
def sum_spectrum(X, s, smax=198):
    s = s.astype(int); D = X.shape[1]
    M = np.zeros((smax+1, D)); c = np.zeros(smax+1)
    np.add.at(M, s, X); np.add.at(c, s, 1)
    keep = c > 0; M = M[keep] - X.mean(0, keepdims=True)
    F = np.abs(np.fft.rfft(M, axis=0)); mag = np.linalg.norm(F, axis=1)
    per = 1.0 / np.maximum(np.fft.rfftfreq(keep.sum()), 1e-9)
    top = np.argsort(mag)[::-1][1:12]
    return [(round(float(per[i]),2), round(float(mag[i]),1)) for i in top]

print("empirical outlier periods @L0:", sum_spectrum(Xres[L0], Sres))

# Richer, independently justified span: empirical periods + small magnitude basis (top PCA dirs ~ sum)
REAL_PERIODS = [2, 2.5, 5, 10, 20, 100]

def mag_basis(X, s, k=4):
    Xc = X - X.mean(0, keepdims=True)
    from sklearn.decomposition import PCA
    pca = PCA(n_components=30, random_state=0)
    Z = pca.fit_transform(Xc)
    corr = np.array([abs(np.corrcoef(Z[:, j], s)[0, 1]) for j in range(30)])
    top = np.argsort(corr)[::-1][:k]
    comps = pca.components_[top]
    Q, _ = np.linalg.qr(comps.T)
    return torch.tensor(Q, dtype=torch.float16, device=model.device)

Hfull = {
    l: orthobasis([fit_Q(Xres[l], Sres, periods=[T]) for T in REAL_PERIODS]
                  + [fit_lin_Q(Xres[l], Sres), mag_basis(Xres[l], Sres)])
    for l in Hspan
}

print("\n" + "=" * 100)
print("Hfull wrong-frequency controls")
print("=" * 100)

hfull_rows = []
for T in [3, 7]:
    Qw = {l: fit_Q(Xres[l], Sres, periods=[T]) for l in Hfull}
    Qwo = {l: orth_to(Qw[l], Hfull[l])[0] for l in Hfull}
    r_raw = eval_persistent_err(eval_examples, Qw)
    r_orth = eval_persistent_err(eval_examples, Qwo)
    print(f"T{T}: raw_exact={r_raw['exact']:.3f} raw_m10={r_raw['mod10']:.3f} | "
          f"orth_to_FULL_exact={r_orth['exact']:.3f} orth_to_FULL_m10={r_orth['mod10']:.3f}")
    hfull_rows.append({"freq": T, "condition": "raw", **r_raw})
    hfull_rows.append({"freq": T, "condition": "orth_to_FULL", **r_orth})

hfull_df = pd.DataFrame(hfull_rows)
save_df(hfull_df, "test4_hfull_wrong_frequency_controls.csv")
print("RANDOM rank-2 anchor:", r_rand2['exact'])




In [ ]:
# ============================================================
# TEST 5: FREQUENCY -> RESIDUE MATRIX
# GPT-J parity: includes T100, LINEAR, T100_LINEAR.
# ============================================================

print("\n" + "=" * 100)
print("TEST 5: FREQUENCY -> RESIDUE MATRIX")
print("=" * 100)

CHANCE = {2: 0.5, 3: 1/3, 5: 0.2, 7: 1/7, 10: 0.1}
KS = (2, 3, 5, 7, 10)


def selectivity(r, ks=KS):
    return {k: (r[f"mod{k}"] - CHANCE[k]) / (1 - CHANCE[k]) for k in ks}

matrix_rows = []
print("RAW residue matrix")
print(f"{'ablate':14} {'exact':>7} " + " ".join(f"{'mod'+str(k):>7}" for k in KS))
print(f"{'none':14} {1.0:>7.3f} " + " ".join(f"{1.0:>7.3f}" for _ in KS))

for name, periods, include_linear in [
    ("T2", [2], False),
    ("T3", [3], False),
    ("T5", [5], False),
    ("T7", [7], False),
    ("T10", [10], False),
    ("T100", [100], False),
    ("LINEAR", [], True),
    ("T100_LINEAR", [100], True),
]:
    if name == "LINEAR":
        Q = {l: fit_lin_Q(Xres[l], Sres) for l in layers if l >= L0}
    else:
        Q = {
            l: fit_Q(Xres[l], Sres, periods=periods, include_linear=include_linear)
            for l in layers if l >= L0
        }

    r = eval_persistent(eval_examples, Q, label=f"matrix {name}", ks=KS)
    row = {"ablate": name, **r}
    row.update({f"sel_mod{k}": selectivity(r)[k] for k in KS})
    matrix_rows.append(row)
    print(f"{name:14} {r['exact']:>7.3f} " + " ".join(f"{r[f'mod{k}']:>7.3f}" for k in KS))

for T in [3, 7]:
    Qp = {
        l: orth_to(fit_Q(Xres[l], Sres, periods=[T], include_linear=False), Hfull[l])[0]
        for l in Hfull
    }
    r = eval_persistent(eval_examples, Qp, label=f"matrix T{T}_perp", ks=KS)
    row = {"ablate": f"T{T}_perp", **r}
    row.update({f"sel_mod{k}": selectivity(r)[k] for k in KS})
    matrix_rows.append(row)
    print(f"{'T'+str(T)+'_perp':14} {r['exact']:>7.3f} " + " ".join(f"{r[f'mod{k}']:>7.3f}" for k in KS))

matrix_df = pd.DataFrame(matrix_rows)
save_df(matrix_df, "test5_residue_matrix.csv")

print("\nNORMALIZED selectivity (acc-chance)/(1-chance) [lower = more damaged]")
print(f"{'ablate':14} " + " ".join(f"{'mod'+str(k):>7}" for k in KS))
for _, row in matrix_df.iterrows():
    print(f"{row['ablate']:14} " + " ".join(f"{row[f'sel_mod{k}']:>7.3f}" for k in KS))

In [ ]:
# ============================================================
# TEST 6: LOW-PERIOD RESIDUE STEERING
#
# Purpose:
#   Test whether the low-period T2/T5/T10 Fourier span can steer
#   the model's output residue / units digit.
#
# This is the sufficiency-style counterpart to ablation:
#
#   Ablation:
#       "Is this subspace necessary?"
#
#   Steering:
#       "Can this subspace push the answer toward a counterfactual residue?"
#
# Core checks:
#   delta=1:
#       T2/T5/T10 steering should push output mod10 toward (sum + 1) mod10.
#
#   delta=2:
#       mod2 is unchanged, mod5/mod10 change.
#
#   delta=5:
#       mod5 is unchanged, mod2/mod10 change.
#
#   delta=10:
#       null-ish control for T2/T5/T10, because s and s+10 have
#       identical residues mod2, mod5, and mod10.
#
# Requires existing globals:
#   model, tokenizer, eval_examples, Xres, Sres, layers, L0
#   get_last_positions, pred_token_to_int, save_df
# ============================================================

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.linear_model import Ridge

print("\n" + "=" * 100)
print("TEST 6: LOW-PERIOD RESIDUE STEERING")
print("=" * 100)


# ------------------------------------------------------------
# Block compatibility: Pythia or GPT-J
# ------------------------------------------------------------

def get_block_list_for_residue_steering():
    if "blocks" in globals():
        return blocks, "blocks"
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        return model.gpt_neox.layers, "model.gpt_neox.layers"
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h, "model.transformer.h"
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers, "model.model.layers"
    raise RuntimeError("Could not find transformer block list.")

RESID_STEER_BLOCKS, RESID_STEER_BLOCK_PATH = get_block_list_for_residue_steering()
print("using block path:", RESID_STEER_BLOCK_PATH)


# ------------------------------------------------------------
# Basis and regression helpers
# ------------------------------------------------------------

def make_residue_steer_basis(sums, periods):
    s = np.asarray(sums, dtype=np.float64)
    cols = []

    for T in periods:
        theta = 2.0 * np.pi * s / float(T)
        cols.append(np.cos(theta))
        cols.append(np.sin(theta))

    if len(cols) == 0:
        raise ValueError("Residue steering basis is empty.")

    return np.column_stack(cols)


def fit_QWB_residue_steer(X, sums, periods):
    """
    Fit:
        X ≈ Y @ W.T + bias

    Returns:
        Q    : orthonormal basis of learned span [D, K]
        W    : regression directions [D, K]
        bias : intercept [D]
    """
    Y = make_residue_steer_basis(sums, periods=periods)

    reg = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
    reg.fit(Y, X)

    W = reg.coef_       # [D, K]
    b = reg.intercept_  # [D]

    Q, _ = np.linalg.qr(W)

    return (
        torch.tensor(Q, dtype=torch.float16, device=model.device),
        torch.tensor(W, dtype=torch.float16, device=model.device),
        torch.tensor(b, dtype=torch.float16, device=model.device),
    )


# ------------------------------------------------------------
# Persistent pin steering hook
# ------------------------------------------------------------

_resid_lp = None
_resid_bi = None
_resid_Q = {}
_resid_target = {}

def make_residue_pin_hook(layer):
    def hook(module, inputs, output):
        global _resid_lp, _resid_bi, _resid_Q, _resid_target

        if isinstance(output, tuple):
            hs = output[0].clone()
            rest = tuple(output[1:])
        else:
            hs = output.clone()
            rest = None

        v = hs[_resid_bi, _resid_lp]
        Q = _resid_Q[layer]

        # Replace current content in Q span with target fitted content.
        hs[_resid_bi, _resid_lp] = v - (v @ Q) @ Q.T + _resid_target[layer]

        return hs if rest is None else (hs,) + rest

    return hook


@torch.no_grad()
def eval_pin_steer_residue(
    rows,
    QWB_by_layer,
    delta,
    periods,
    alpha=1.0,
    ks=(2, 5, 10, 100),
    label="",
):
    """
    Pin steering for low-period residue subspaces.

    alpha:
      0.0 -> pin to fitted current content for s
      1.0 -> pin to fitted counterfactual content for s + delta
      >1  -> extrapolate

    Metrics:
      follow{k}: pred % k == (true + delta) % k
      stay{k}:   pred % k == true % k

    For delta=10:
      follow2/follow5/follow10 are equal to stay2/stay5/stay10
      by construction, so this should be null-ish for T2/T5/T10.
    """
    global _resid_lp, _resid_bi, _resid_Q, _resid_target

    handles = [
        RESID_STEER_BLOCKS[l].register_forward_hook(make_residue_pin_hook(l))
        for l in QWB_by_layer
    ]

    n = 0
    exact_original = 0
    exact_delta = 0
    parseable = 0

    follow = {k: 0 for k in ks}
    stay = {k: 0 for k in ks}

    same_units = 0
    same_decade = 0

    shifts = []

    try:
        for start in tqdm(range(0, len(rows), BATCH_SIZE), desc=f"residue steer {label}"):
            batch = rows[start:start + BATCH_SIZE]

            s = np.array([x["sum"] for x in batch], dtype=np.float64)
            s_target = s + delta

            Ys = make_residue_steer_basis(s, periods=periods)
            Yt = make_residue_steer_basis(s_target, periods=periods)

            Yint_np = Ys + alpha * (Yt - Ys)
            Yint = torch.tensor(Yint_np, dtype=torch.float16, device=model.device)

            enc = tokenizer(
                [x["prompt"] for x in batch],
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            ).to(model.device)

            _resid_lp = get_last_positions(enc["attention_mask"])
            _resid_bi = torch.arange(len(batch), device=model.device)

            _resid_Q = {}
            _resid_target = {}

            for l, (Q, W, bias) in QWB_by_layer.items():
                yhat = Yint @ W.T + bias
                target_q = (yhat @ Q) @ Q.T

                _resid_Q[l] = Q
                _resid_target[l] = target_q

            logits = model(**enc, use_cache=False).logits[_resid_bi, _resid_lp]
            pred_ids = logits.argmax(-1).detach().cpu().tolist()

            for pred_id, ex in zip(pred_ids, batch):
                n += 1

                true_sum = int(ex["sum"])
                target_sum = int(true_sum + delta)

                exact_original += int(int(pred_id) == int(ex["target_token_id"]))

                pred_num = pred_token_to_int(pred_id)
                if pred_num is None:
                    continue

                parseable += 1
                exact_delta += int(pred_num == target_sum)

                shift = pred_num - true_sum
                shifts.append(shift)

                same_units += int(pred_num % 10 == true_sum % 10)
                same_decade += int(pred_num // 10 == true_sum // 10)

                for k in ks:
                    follow[k] += int(pred_num % k == target_sum % k)
                    stay[k] += int(pred_num % k == true_sum % k)

    finally:
        for h in handles:
            h.remove()

    n_safe = max(n, 1)

    out = {
        "label": label,
        "delta": delta,
        "alpha": alpha,
        "periods": str(periods),
        "n": n,
        "exact_original": exact_original / n_safe,
        "exact_delta": exact_delta / n_safe,
        "parseable": parseable / n_safe,
        "same_units": same_units / n_safe,
        "same_decade": same_decade / n_safe,
        "median_shift": float(np.median(shifts)) if len(shifts) else None,
        "mean_shift": float(np.mean(shifts)) if len(shifts) else None,
        "median_abs_shift": float(np.median(np.abs(shifts))) if len(shifts) else None,
    }

    for k in ks:
        out[f"follow{k}"] = follow[k] / n_safe
        out[f"stay{k}"] = stay[k] / n_safe
        out[f"net{k}"] = (follow[k] - stay[k]) / n_safe

    return out


# ------------------------------------------------------------
# Fit low-period steering spans
# ------------------------------------------------------------

print("\nFitting low-period residue steering spans...")

RESID_STEER_LAYER_SET = [l for l in layers if l >= L0]
print("steering layers:", RESID_STEER_LAYER_SET)

QWB_LOW_T2T5T10 = {
    l: fit_QWB_residue_steer(Xres[l], Sres, periods=[2, 5, 10])
    for l in RESID_STEER_LAYER_SET
}

QWB_T5_ONLY = {
    l: fit_QWB_residue_steer(Xres[l], Sres, periods=[5])
    for l in RESID_STEER_LAYER_SET
}

QWB_T10_ONLY = {
    l: fit_QWB_residue_steer(Xres[l], Sres, periods=[10])
    for l in RESID_STEER_LAYER_SET
}

print("done fitting low-period residue steering spans.")


# ------------------------------------------------------------
# Run steering tests
# ------------------------------------------------------------

# Full run uses all baseline-correct eval examples.
# To debug faster, set:
#   RESID_STEER_EVAL = eval_examples[:1000]
RESID_STEER_EVAL = eval_examples

residue_steer_rows = []

def run_residue_condition(mode, QWB, periods, deltas, alphas):
    print(f"\n{mode}")
    print(
        f"{'delta':>5} {'alpha':>6} {'orig':>7} {'target':>7} "
        f"{'med_shift':>10} {'sameU':>7} {'sameD':>7} "
        f"{'follow10':>8} {'stay10':>7} {'net10':>7} "
        f"{'follow5':>8} {'stay5':>7} {'net5':>7} "
        f"{'follow2':>8} {'stay2':>7} {'net2':>7}"
    )
    print("-" * 150)

    for delta in deltas:
        for alpha in alphas:
            r = eval_pin_steer_residue(
                RESID_STEER_EVAL,
                QWB,
                delta=delta,
                periods=periods,
                alpha=alpha,
                ks=(2, 5, 10, 100),
                label=f"{mode}_delta{delta}_alpha{alpha}",
            )
            r["mode"] = mode
            residue_steer_rows.append(r)

            print(
                f"{delta:>5} {alpha:>6.2f} "
                f"{r['exact_original']:>7.3f} {r['exact_delta']:>7.3f} "
                f"{(r['median_shift'] if r['median_shift'] is not None else 0):>10.1f} "
                f"{r['same_units']:>7.3f} {r['same_decade']:>7.3f} "
                f"{r['follow10']:>8.3f} {r['stay10']:>7.3f} {r['net10']:>7.3f} "
                f"{r['follow5']:>8.3f} {r['stay5']:>7.3f} {r['net5']:>7.3f} "
                f"{r['follow2']:>8.3f} {r['stay2']:>7.3f} {r['net2']:>7.3f}"
            )


# Main low-period steering:
# delta=1: all low-period residues change.
# delta=2: parity unchanged, mod5/mod10 change.
# delta=5: mod5 unchanged, parity/mod10 change.
# delta=10: null-ish for T2/T5/T10.
run_residue_condition(
    mode="LOW_T2T5T10",
    QWB=QWB_LOW_T2T5T10,
    periods=[2, 5, 10],
    deltas=[1, 2, 5, 10],
    alphas=[0.0, 0.5, 1.0, 1.5],
)

# T5-only should mainly affect mod5/mod10, not mod2.
run_residue_condition(
    mode="T5_ONLY",
    QWB=QWB_T5_ONLY,
    periods=[5],
    deltas=[1, 2],
    alphas=[0.0, 0.5, 1.0, 1.5],
)

# T10-only is a direct units-digit plane.
run_residue_condition(
    mode="T10_ONLY",
    QWB=QWB_T10_ONLY,
    periods=[10],
    deltas=[1, 5, 10],
    alphas=[0.0, 0.5, 1.0, 1.5],
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

residue_steer_df = pd.DataFrame(residue_steer_rows)
save_df(residue_steer_df, "test6_low_period_residue_steering.csv")

print("\nInterpretation checklist:")
print("  - LOW_T2T5T10 delta=1 should increase follow10 over stay10 if residue steering works.")
print("  - LOW_T2T5T10 delta=10 should be null-ish for mod2/mod5/mod10 because residues are unchanged.")
print("  - T5_ONLY should affect follow5/follow10 more than follow2.")
print("  - T10_ONLY should directly steer units digit; delta=10 should be null-ish.")
print("  - exact_delta is strict; follow10/follow5/follow2 are usually more informative.")

In [ ]:
# ============================================================
# TEST 7: T100 / LINEAR / MAGNITUDE STEERING
#
# Purpose:
#   Test whether long-period / linear channels can steer coarse magnitude
#   while preserving units digit.
#
# Core idea:
#   Low-period T2/T5/T10 cannot distinguish s from s+10.
#   T100 / LINEAR should be able to distinguish neighboring decades.
#
# Best expected pattern:
#   LOW_T2T5T10, delta=10:
#       null-ish; follow100 should not strongly beat stay100.
#   T100 or T100_LINEAR, delta=10:
#       same_units should remain high,
#       follow100 should increase over stay100,
#       median_shift should move toward +10.
#
# Requires existing globals:
#   model, tokenizer, eval_examples, Xres, Sres, layers, L0
#   get_last_positions, pred_token_to_int, save_df
# ============================================================

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.linear_model import Ridge

print("\n" + "=" * 100)
print("TEST 7: T100 / LINEAR / MAGNITUDE STEERING")
print("=" * 100)


# ------------------------------------------------------------
# Block compatibility: Pythia or GPT-J
# ------------------------------------------------------------

def get_block_list_for_steering():
    if "blocks" in globals():
        return blocks, "blocks"
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        return model.gpt_neox.layers, "model.gpt_neox.layers"
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h, "model.transformer.h"
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers, "model.model.layers"
    raise RuntimeError("Could not find transformer block list.")

STEER_BLOCKS, STEER_BLOCK_PATH = get_block_list_for_steering()
print("using block path:", STEER_BLOCK_PATH)


# ------------------------------------------------------------
# Fixed-basis builder
#
# Important:
#   For steering with a linear term, we must use the same normalization
#   at fit-time and intervention-time. Do NOT recompute mean/std per batch.
# ------------------------------------------------------------

STEER_LINEAR_MEAN = float(np.mean(Sres))
STEER_LINEAR_STD = float(np.std(Sres) + 1e-12)


def make_steer_basis(sums, periods, include_linear=False):
    s = np.asarray(sums, dtype=np.float64)
    cols = []

    if include_linear:
        z = (s - STEER_LINEAR_MEAN) / STEER_LINEAR_STD
        cols.append(z)

    for T in periods:
        theta = 2.0 * np.pi * s / float(T)
        cols.append(np.cos(theta))
        cols.append(np.sin(theta))

    if len(cols) == 0:
        raise ValueError("Basis is empty. Use include_linear=True or provide at least one period.")

    return np.column_stack(cols)


def fit_QWB_steer(X, sums, periods, include_linear=False):
    """
    Fit X ≈ Y @ W.T + bias.

    Returns:
      Q    : orthonormal basis of learned span [D, K]
      W    : regression directions [D, K]
      bias : intercept [D]
    """
    Y = make_steer_basis(sums, periods=periods, include_linear=include_linear)

    reg = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
    reg.fit(Y, X)

    W = reg.coef_       # [D, K]
    b = reg.intercept_  # [D]

    Q, _ = np.linalg.qr(W)

    return (
        torch.tensor(Q, dtype=torch.float16, device=model.device),
        torch.tensor(W, dtype=torch.float16, device=model.device),
        torch.tensor(b, dtype=torch.float16, device=model.device),
    )


# ------------------------------------------------------------
# Persistent pin steering hook
# ------------------------------------------------------------

_mag_lp = None
_mag_bi = None
_mag_Q = {}
_mag_target = {}


def make_magnitude_pin_hook(layer):
    def hook(module, inputs, output):
        global _mag_lp, _mag_bi, _mag_Q, _mag_target

        if isinstance(output, tuple):
            hs = output[0].clone()
            rest = tuple(output[1:])
        else:
            hs = output.clone()
            rest = None

        v = hs[_mag_bi, _mag_lp]
        Q = _mag_Q[layer]

        # Replace current content in Q span with target content.
        hs[_mag_bi, _mag_lp] = v - (v @ Q) @ Q.T + _mag_target[layer]

        return hs if rest is None else (hs,) + rest

    return hook


@torch.no_grad()
def eval_pin_steer_magnitude(
    rows,
    QWB_by_layer,
    delta,
    periods,
    include_linear=False,
    alpha=1.0,
    ks=(2, 5, 10, 100),
    label="",
):
    """
    General pin steering for T100 / LINEAR / T100_LINEAR / FULL.

    alpha:
      0.0 -> pin to fitted current content for s
      1.0 -> pin to fitted counterfactual content for s + delta
      >1  -> extrapolate

    Metrics:
      follow{k}: pred % k == (true + delta) % k
      stay{k}:   pred % k == true % k

    For delta=10:
      follow10 == stay10 by definition, so mod100 / exact_delta / median_shift
      are the important metrics.
    """
    global _mag_lp, _mag_bi, _mag_Q, _mag_target

    handles = [
        STEER_BLOCKS[l].register_forward_hook(make_magnitude_pin_hook(l))
        for l in QWB_by_layer
    ]

    n = 0
    exact_original = 0
    exact_delta = 0
    parseable = 0

    follow = {k: 0 for k in ks}
    stay = {k: 0 for k in ks}

    same_units = 0
    same_decade = 0

    shifts = []

    try:
        for start in tqdm(range(0, len(rows), BATCH_SIZE), desc=f"mag steer {label}"):
            batch = rows[start:start + BATCH_SIZE]

            s = np.array([x["sum"] for x in batch], dtype=np.float64)
            s_target = s + delta

            Ys = make_steer_basis(s, periods=periods, include_linear=include_linear)
            Yt = make_steer_basis(s_target, periods=periods, include_linear=include_linear)

            Yint_np = Ys + alpha * (Yt - Ys)
            Yint = torch.tensor(Yint_np, dtype=torch.float16, device=model.device)

            enc = tokenizer(
                [x["prompt"] for x in batch],
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            ).to(model.device)

            _mag_lp = get_last_positions(enc["attention_mask"])
            _mag_bi = torch.arange(len(batch), device=model.device)

            _mag_Q = {}
            _mag_target = {}

            for l, (Q, W, bias) in QWB_by_layer.items():
                yhat = Yint @ W.T + bias
                target_q = (yhat @ Q) @ Q.T

                _mag_Q[l] = Q
                _mag_target[l] = target_q

            logits = model(**enc, use_cache=False).logits[_mag_bi, _mag_lp]
            pred_ids = logits.argmax(-1).detach().cpu().tolist()

            for pred_id, ex in zip(pred_ids, batch):
                n += 1

                true_sum = int(ex["sum"])
                target_sum = int(true_sum + delta)

                exact_original += int(int(pred_id) == int(ex["target_token_id"]))

                pred_num = pred_token_to_int(pred_id)
                if pred_num is None:
                    continue

                parseable += 1
                exact_delta += int(pred_num == target_sum)

                shift = pred_num - true_sum
                shifts.append(shift)

                same_units += int(pred_num % 10 == true_sum % 10)
                same_decade += int(pred_num // 10 == true_sum // 10)

                for k in ks:
                    follow[k] += int(pred_num % k == target_sum % k)
                    stay[k] += int(pred_num % k == true_sum % k)

    finally:
        for h in handles:
            h.remove()

    n_safe = max(n, 1)

    out = {
        "label": label,
        "delta": delta,
        "alpha": alpha,
        "periods": str(periods),
        "include_linear": include_linear,
        "n": n,
        "exact_original": exact_original / n_safe,
        "exact_delta": exact_delta / n_safe,
        "parseable": parseable / n_safe,
        "same_units": same_units / n_safe,
        "same_decade": same_decade / n_safe,
        "median_shift": float(np.median(shifts)) if len(shifts) else None,
        "mean_shift": float(np.mean(shifts)) if len(shifts) else None,
        "median_abs_shift": float(np.median(np.abs(shifts))) if len(shifts) else None,
    }

    for k in ks:
        out[f"follow{k}"] = follow[k] / n_safe
        out[f"stay{k}"] = stay[k] / n_safe
        out[f"net{k}"] = (follow[k] - stay[k]) / n_safe

    return out


# ------------------------------------------------------------
# Fit steering spans
# ------------------------------------------------------------

print("\nFitting steering spans...")

STEER_LAYER_SET = [l for l in layers if l >= L0]
print("steering layers:", STEER_LAYER_SET)

QWB_LOW = {
    l: fit_QWB_steer(Xres[l], Sres, periods=[2, 5, 10], include_linear=False)
    for l in STEER_LAYER_SET
}

QWB_T100 = {
    l: fit_QWB_steer(Xres[l], Sres, periods=[100], include_linear=False)
    for l in STEER_LAYER_SET
}

QWB_LINEAR = {
    l: fit_QWB_steer(Xres[l], Sres, periods=[], include_linear=True)
    for l in STEER_LAYER_SET
}

QWB_T100_LINEAR = {
    l: fit_QWB_steer(Xres[l], Sres, periods=[100], include_linear=True)
    for l in STEER_LAYER_SET
}

QWB_FULL = {
    l: fit_QWB_steer(Xres[l], Sres, periods=[2, 5, 10, 100], include_linear=True)
    for l in STEER_LAYER_SET
}

print("done fitting steering spans.")


# ------------------------------------------------------------
# Run steering tests
# ------------------------------------------------------------

# Full paper run uses all baseline-correct eval examples.
# To debug faster, set for example:
#   MAG_STEER_EVAL = eval_examples[:1000]
MAG_STEER_EVAL = eval_examples

mag_rows = []


def run_mag_condition(mode, QWB, periods, include_linear, deltas, alphas):
    print(f"\n{mode}")
    print(
        f"{'delta':>5} {'alpha':>6} {'orig':>7} {'target':>7} "
        f"{'med_shift':>10} {'sameU':>7} {'sameD':>7} "
        f"{'follow100':>9} {'stay100':>8} {'net100':>8} "
        f"{'follow10':>8} {'stay10':>7}"
    )
    print("-" * 110)

    for delta in deltas:
        for alpha in alphas:
            r = eval_pin_steer_magnitude(
                MAG_STEER_EVAL,
                QWB,
                delta=delta,
                periods=periods,
                include_linear=include_linear,
                alpha=alpha,
                ks=(2, 5, 10, 100),
                label=f"{mode}_delta{delta}_alpha{alpha}",
            )
            r["mode"] = mode
            mag_rows.append(r)

            print(
                f"{delta:>5} {alpha:>6.2f} "
                f"{r['exact_original']:>7.3f} {r['exact_delta']:>7.3f} "
                f"{(r['median_shift'] if r['median_shift'] is not None else 0):>10.1f} "
                f"{r['same_units']:>7.3f} {r['same_decade']:>7.3f} "
                f"{r['follow100']:>9.3f} {r['stay100']:>8.3f} {r['net100']:>8.3f} "
                f"{r['follow10']:>8.3f} {r['stay10']:>7.3f}"
            )


# 1) Null-ish control:
#    Low-period channel cannot distinguish s from s+10.
run_mag_condition(
    mode="LOW_T2T5T10",
    QWB=QWB_LOW,
    periods=[2, 5, 10],
    include_linear=False,
    deltas=[10],
    alphas=[0.0, 0.5, 1.0, 1.5],
)

# 2) T100 should distinguish s from s+10/s+20/s+50 modulo 100.
run_mag_condition(
    mode="T100_ONLY",
    QWB=QWB_T100,
    periods=[100],
    include_linear=False,
    deltas=[10, 20, 50],
    alphas=[0.0, 0.5, 1.0, 1.5],
)

# 3) Linear magnitude steering.
run_mag_condition(
    mode="LINEAR_ONLY",
    QWB=QWB_LINEAR,
    periods=[],
    include_linear=True,
    deltas=[10, 20, 50],
    alphas=[0.0, 0.5, 1.0, 1.5],
)

# 4) T100 + Linear combined magnitude channel.
run_mag_condition(
    mode="T100_LINEAR",
    QWB=QWB_T100_LINEAR,
    periods=[100],
    include_linear=True,
    deltas=[10, 20, 50],
    alphas=[0.0, 0.5, 1.0, 1.5],
)

# 5) Full low-period + magnitude helix.
run_mag_condition(
    mode="FULL_T2T5T10T100_LINEAR",
    QWB=QWB_FULL,
    periods=[2, 5, 10, 100],
    include_linear=True,
    deltas=[10, 20],
    alphas=[0.0, 0.5, 1.0, 1.5],
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

mag_steer_df = pd.DataFrame(mag_rows)
save_df(mag_steer_df, "test7_t100_linear_magnitude_steering.csv")

print("\nInterpretation checklist:")
print("  - LOW_T2T5T10 delta=10 should be null-ish: follow100 should not strongly beat stay100.")
print("  - T100_ONLY / T100_LINEAR delta=10 should increase follow100 vs stay100 if they steer decade/magnitude.")
print("  - same_units should stay high for magnitude steering if only decade/localization changes.")
print("  - exact_delta is a strict metric; follow100 + median_shift may be more informative.")
print("  - If LINEAR produces huge shifts/noisy outputs, interpret as magnitude channel but not precise steering.")


In [ ]:
# ============================================================
# HUMAN RESULTS DASHBOARD
#
# Purpose:
#   Read saved CSVs and print human-readable summary tables.
#
# Requires:
#   OUT_DIR
#
# Reads if available:
#   baseline_predictions.csv
#   test1_persistent_sweep.csv
#   test2_periodwise_ablation.csv
#   extra_decade_same_units_analysis.csv
#   test3_wrong_frequency_orthogonalized.csv
#   test4_hfull_wrong_frequency_controls.csv
#   test5_residue_matrix.csv
#   test6_low_period_residue_steering.csv
#   test7_t100_linear_magnitude_steering.csv
# ============================================================

import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

print("\n" + "=" * 100)
print("HUMAN RESULTS DASHBOARD")
print("=" * 100)
print("OUT_DIR:", OUT_DIR)

def read_csv_if_exists(name):
    path = os.path.join(OUT_DIR, name)

    if not os.path.exists(path):
        print(f"missing: {name}")
        return None

    if os.path.getsize(path) == 0:
        print(f"empty: {name}")
        return None

    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        print(f"empty/unparseable: {name}")
        return None

    if len(df.columns) == 0:
        print(f"empty columns: {name}")
        return None

    print(f"loaded: {name} ({len(df)} rows)")
    return df

def pct(x):
    if pd.isna(x):
        return ""
    return f"{100 * float(x):.1f}%"

def num(x, nd=3):
    if pd.isna(x):
        return ""
    return f"{float(x):.{nd}f}"

def show(title, df, max_rows=50):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    if df is None or len(df) == 0:
        print("(empty / missing)")
        return
    print(df.head(max_rows).to_string(index=False))

def save_table(df, name):
    path = os.path.join(OUT_DIR, name)
    df.to_csv(path, index=False)
    print("saved:", path)

baseline_df = read_csv_if_exists("baseline_predictions.csv")
sweep_df = read_csv_if_exists("test1_persistent_sweep.csv")
periodwise_df_saved = read_csv_if_exists("test2_periodwise_ablation.csv")
decade_df_saved = read_csv_if_exists("extra_decade_same_units_analysis.csv")
wrong_df_saved = read_csv_if_exists("test3_wrong_frequency_orthogonalized.csv")
hfull_df_saved = read_csv_if_exists("test4_hfull_wrong_frequency_controls.csv")
matrix_df_saved = read_csv_if_exists("test5_residue_matrix.csv")
steer6_df = read_csv_if_exists("test6_low_period_residue_steering.csv")
steer7_df = read_csv_if_exists("test7_t100_linear_magnitude_steering.csv")

human_tables = {}
markdown_lines = []

def md(line=""):
    markdown_lines.append(line)

md("# Human-readable results summary")
md("")
md(f"Output directory: `{OUT_DIR}`")
md("")

# ============================================================
# 0. BASELINE / NATURAL ERRORS
# ============================================================

if baseline_df is not None:
    n_total = len(baseline_df)
    n_correct = int(baseline_df["correct"].sum())
    exact = n_correct / max(n_total, 1)

    wrong = baseline_df[~baseline_df["correct"].astype(bool)].copy()
    parse_wrong = wrong[wrong["pred"].notna()].copy()

    rows = [{
        "model_examples": n_total,
        "baseline_correct": n_correct,
        "baseline_exact": pct(exact),
        "wrong_total": len(wrong),
        "parseable_wrong": len(parse_wrong),
        "unparseable_wrong": len(wrong) - len(parse_wrong),
    }]

    baseline_summary = pd.DataFrame(rows)
    human_tables["00_baseline_summary"] = baseline_summary
    show("0A. BASELINE SUMMARY", baseline_summary)

    md("## 0. Baseline")
    md("")
    md(f"- Baseline exact accuracy: **{pct(exact)}** ({n_correct}/{n_total}).")
    md(f"- Parseable wrong answers: **{len(parse_wrong)}**.")
    md("")

    if len(parse_wrong):
        parse_wrong["diff"] = parse_wrong["pred"].astype(int) - parse_wrong["sum"].astype(int)

        top_shifts = (
            parse_wrong["diff"]
            .value_counts()
            .head(12)
            .rename_axis("pred_minus_true")
            .reset_index(name="count")
        )
        top_shifts["percent_of_parseable_wrong"] = top_shifts["count"] / len(parse_wrong)
        top_shifts["percent_of_parseable_wrong"] = top_shifts["percent_of_parseable_wrong"].map(pct)

        human_tables["01_top_natural_error_shifts"] = top_shifts
        show("0B. TOP NATURAL ERROR SHIFTS", top_shifts)

        residue_rows = []
        for k in [2, 3, 5, 7, 10]:
            keep = np.mean(parse_wrong["pred"].astype(int) % k == parse_wrong["sum"].astype(int) % k)
            residue_rows.append({
                "residue": f"mod{k}",
                "preserved_among_parseable_wrong": pct(keep),
            })
        residue_summary = pd.DataFrame(residue_rows)

        human_tables["02_natural_error_residue_preservation"] = residue_summary
        show("0C. NATURAL ERROR RESIDUE PRESERVATION", residue_summary)

        pm10 = np.mean(parse_wrong["diff"].isin([-10, 10]))
        mult10 = np.mean(np.abs(parse_wrong["diff"]) % 10 == 0)

        md(f"- Among parseable wrong answers, **±10 errors**: **{pct(pm10)}**.")
        md(f"- Among parseable wrong answers, **units digit preserved / multiple-of-10 error**: **{pct(mult10)}**.")
        md("")

# ============================================================
# 1. PERSISTENT SWEEP
# ============================================================

if sweep_df is not None:
    pivot_rows = []
    for L in sorted(sweep_df["L_start"].unique(), reverse=True):
        h = sweep_df[(sweep_df["L_start"] == L) & (sweep_df["condition"].astype(str).str.contains("helix"))]
        r = sweep_df[(sweep_df["L_start"] == L) & (sweep_df["condition"].astype(str).str.contains("random"))]

        if len(h) and len(r):
            h = h.iloc[0]
            r = r.iloc[0]
            pivot_rows.append({
                "L_start": int(L),
                "n_layers": int(h["n_layers"]),
                "T2T5T10_exact": pct(h["exact"]),
                "random_exact": pct(r["exact"]),
                "T2T5T10_mod10": pct(h["mod10"]),
                "random_mod10": pct(r["mod10"]),
                "exact_gap_random_minus_T2T5T10": pct(float(r["exact"]) - float(h["exact"])),
                "mod10_gap_random_minus_T2T5T10": pct(float(r["mod10"]) - float(h["mod10"])),
            })

    sweep_human = pd.DataFrame(pivot_rows)
    human_tables["03_persistent_sweep"] = sweep_human
    show("1. PERSISTENT T2/T5/T10 ABLATION SWEEP", sweep_human)

    md("## 1. Persistent low-period ablation sweep")
    md("")
    if len(sweep_human):
        best = sweep_human.iloc[-1]
        md("- T2/T5/T10 persistent ablation is compared against matched-rank random ablation across start layers.")
        md("- Read this as: if T2/T5/T10 collapses but random does not, the effect is subspace-specific rather than rank damage.")
        md("")

# ============================================================
# 2. PERIOD-WISE ABLATION
# ============================================================

if periodwise_df_saved is not None:
    want = [
        "T2", "T5", "T10", "T100", "LINEAR", "T100_LINEAR",
        "T2T5T10", "T2T5T10T100", "RANDOM_rank6",
        "T3_ctrl", "T7_ctrl",
    ]

    rows = []
    for name in want:
        sub = periodwise_df_saved[periodwise_df_saved["subspace"] == name]
        if len(sub) == 0:
            continue

        if name == "RANDOM_rank6":
            row = sub[["exact", "mod2", "mod3", "mod5", "mod7", "mod10", "parseable"]].mean().to_dict()
            label = "RANDOM_rank6 mean"
        else:
            row = sub.iloc[0].to_dict()
            label = name

        rows.append({
            "condition": label,
            "exact": pct(row.get("exact")),
            "mod2": pct(row.get("mod2")),
            "mod3": pct(row.get("mod3")),
            "mod5": pct(row.get("mod5")),
            "mod7": pct(row.get("mod7")),
            "mod10": pct(row.get("mod10")),
            "parseable": pct(row.get("parseable")),
            "median_abs_err_wrong": num(row.get("median_abs_err_wrong"), 1),
            "wrong_units_frac": pct(row.get("frac_wrong_units")),
            "human_read": {
                "T2T5T10": "low-period residue channel removed",
                "T100": "long-period / coarse position removed",
                "LINEAR": "linear magnitude removed",
                "T100_LINEAR": "combined magnitude channel removed",
                "RANDOM_rank6 mean": "matched-rank random control",
            }.get(label, ""),
        })

    period_human = pd.DataFrame(rows)
    human_tables["04_periodwise_ablation"] = period_human
    show("2. PERIOD-WISE ABLATION HUMAN TABLE", period_human)

    md("## 2. Period-wise ablation")
    md("")
    md("- `condition` tells you what was removed.")
    md("- `exact` is full answer accuracy after removal.")
    md("- `mod10` is units-digit accuracy after removal.")
    md("- If `exact` falls but `mod10` stays high, the model still knows the units digit but lost magnitude/localization.")
    md("")

# ============================================================
# 2B. DECADE / SAME-UNITS
# ============================================================

if decade_df_saved is not None:
    rows = []
    for _, r in decade_df_saved.iterrows():
        rows.append({
            "condition": r["label"],
            "parseable_n": int(r["parseable_n"]),
            "exact_parseable": pct(r.get("exact_parseable")),
            "same_units_all": pct(r.get("same_units_all")),
            "same_decade_all": pct(r.get("same_decade_all")),
            "same_units_wrong": pct(r.get("same_units_wrong")),
            "same_decade_wrong": pct(r.get("same_decade_wrong")),
            "multiple_of_10_wrong": pct(r.get("multiple_of_10_wrong")),
            "pm10_wrong": pct(r.get("pm10_wrong")),
            "median_abs_err_wrong": num(r.get("median_abs_err_wrong"), 1),
            "human_read": (
                "Units preserved but decade destroyed = magnitude/localization channel affected."
                if r.get("same_units_wrong", 0) > 0.8 and r.get("same_decade_wrong", 1) < 0.1
                else "Units destroyed = residue channel affected."
                if r.get("same_units_wrong", 1) < 0.2
                else ""
            ),
        })

    decade_human = pd.DataFrame(rows)
    human_tables["05_decade_same_units"] = decade_human
    show("2B. DECADE / SAME-UNITS ANALYSIS", decade_human)

    md("## 2B. Decade / same-units analysis")
    md("")
    md("- `same_units_wrong`: among wrong parseable outputs, did the output keep the correct units digit?")
    md("- `same_decade_wrong`: among wrong parseable outputs, did the output stay in the correct decade?")
    md("- T100/LINEAR showing high same-units but low same-decade means magnitude/localization is hit while residue remains.")
    md("")

# ============================================================
# 3/4. WRONG-FREQUENCY CONTROLS
# ============================================================

def summarize_hspan_wrong_controls(df):
    """
    New Test 3 schema:
      one row per frequency, with columns:
        raw_exact, orth_exact, raw_mod10, orth_mod10,
        exact_recovery, mod10_recovery

    This function is intentionally only for the tight Hspan control:
      Hspan = [T2, T5, T10, T100, LINEAR]

    It ignores accidental diagnostic/random-anchor rows.
    """
    if df is None or len(df) == 0:
        return None

    required = {"raw_exact", "orth_exact", "raw_mod10", "orth_mod10", "freq"}
    if not required.issubset(set(df.columns)):
        return None

    rows = []

    for _, r in df.iterrows():
        freq = str(r.get("freq"))

        # Only display the actual wrong-frequency controls.
        # This prevents accidental random-anchor / diagnostic rows from appearing.
        if freq not in ["T3", "T7", "3", "7"]:
            continue

        freq_label = freq if freq.startswith("T") else f"T{freq}"

        raw_exact = float(r["raw_exact"])
        orth_exact = float(r["orth_exact"])
        raw_mod10 = float(r["raw_mod10"])
        orth_mod10 = float(r["orth_mod10"])

        rows.append({
            "control_set": "Hspan",
            "freq": freq_label,
            "raw_exact": pct(raw_exact),
            "orth_exact": pct(orth_exact),
            "exact_recovery": pct(orth_exact - raw_exact),
            "raw_mod10": pct(raw_mod10),
            "orth_mod10": pct(orth_mod10),
            "mod10_recovery": pct(orth_mod10 - raw_mod10),
            "cos_L0": r.get("cos_L0", ""),
            "resid_norm_L0": r.get("resid_norm_L0", ""),
            "human_read": (
                "Primary tight control: orthogonalized only against predeclared Hspan "
                "[T2,T5,T10,T100,LINEAR]. Recovery supports leakage/overlap explanation."
            ),
        })

    if len(rows) == 0:
        return None

    return pd.DataFrame(rows)


def summarize_hfull_wrong_controls(df):
    """
    Old Test 4 schema:
      separate rows:
        condition == raw
        condition == orth_to_FULL
      with columns:
        exact, mod10
    """
    if df is None or len(df) == 0:
        return None

    required = {"freq", "condition", "exact", "mod10"}
    if not required.issubset(set(df.columns)):
        return None

    rows = []
    for T in [3, 7]:
        raw = df[
            (df["freq"].astype(str) == str(T)) &
            (df["condition"].astype(str).str.contains("raw", case=False, na=False))
        ]
        orth = df[
            (df["freq"].astype(str) == str(T)) &
            (df["condition"].astype(str).str.contains("orth", case=False, na=False))
        ]

        if len(raw) and len(orth):
            raw = raw.iloc[0]
            orth = orth.iloc[0]

            rows.append({
                "control_set": "Hfull",
                "freq": f"T{T}",
                "raw_exact": pct(raw["exact"]),
                "orth_exact": pct(orth["exact"]),
                "raw_mod10": pct(raw["mod10"]),
                "orth_mod10": pct(orth["mod10"]),
                "exact_recovery": pct(float(orth["exact"]) - float(raw["exact"])),
                "mod10_recovery": pct(float(orth["mod10"]) - float(raw["mod10"])),
                "cos_L0": "",
                "resid_norm_L0": "",
                "human_read": (
                    "Supplementary broader control: orthogonalized against Hfull, including extra empirical/PCA magnitude basis. "
                    "Useful robustness check but not the primary reviewer-facing control."
                ),
            })

    return pd.DataFrame(rows)


wrong_human = summarize_hspan_wrong_controls(wrong_df_saved)
hfull_human = summarize_hfull_wrong_controls(hfull_df_saved)

if wrong_human is not None and len(wrong_human):
    human_tables["06_wrong_frequency_controls_hspan"] = wrong_human
    show("3. WRONG-FREQUENCY ORTHOGONALIZED CONTROLS — Hspan primary", wrong_human)

    md("## 3. Wrong-frequency controls — Hspan primary")
    md("")
    md("- This is the tight reviewer-facing control.")
    md("- T3/T7 are orthogonalized only against the predeclared real helix span: `T2`, `T5`, `T10`, `T100`, and `LINEAR`.")
    md("- If exact/mod10 recover after this orthogonalization, raw T3/T7 damage was mostly overlap/leakage into the real helix span.")
    md("")

if hfull_human is not None and len(hfull_human):
    human_tables["06b_wrong_frequency_controls_hfull"] = hfull_human
    show("4. WRONG-FREQUENCY ORTHOGONALIZED CONTROLS — Hfull supplementary", hfull_human)

    md("## 4. Wrong-frequency controls — Hfull supplementary")
    md("")
    md("- Hfull is a broader robustness control.")
    md("- Because Hfull includes extra empirical/PCA magnitude basis directions, it should not be the primary defense against wrong-frequency objections.")
    md("")

# ============================================================
# 5. RESIDUE MATRIX
# ============================================================

if matrix_df_saved is not None:
    rows = []
    keep = ["T2", "T3", "T5", "T7", "T10", "T100", "LINEAR", "T100_LINEAR", "T3_perp", "T7_perp"]

    for name in keep:
        sub = matrix_df_saved[matrix_df_saved["ablate"] == name]
        if len(sub) == 0:
            continue
        r = sub.iloc[0]

        rows.append({
            "ablate": name,
            "exact": pct(r["exact"]),
            "mod2": pct(r["mod2"]),
            "mod3": pct(r["mod3"]),
            "mod5": pct(r["mod5"]),
            "mod7": pct(r["mod7"]),
            "mod10": pct(r["mod10"]),
            "sel_mod2": num(r.get("sel_mod2")),
            "sel_mod3": num(r.get("sel_mod3")),
            "sel_mod5": num(r.get("sel_mod5")),
            "sel_mod7": num(r.get("sel_mod7")),
            "sel_mod10": num(r.get("sel_mod10")),
        })

    matrix_human = pd.DataFrame(rows)
    human_tables["07_residue_matrix"] = matrix_human
    show("5. RESIDUE MATRIX", matrix_human)

    md("## 5. Residue matrix")
    md("")
    md("- Lower normalized selectivity means that residue class was more damaged.")
    md("- This table is for fingerprinting which Fourier period damages which residue metrics.")
    md("")

# ============================================================
# 6. LOW-PERIOD RESIDUE STEERING
# ============================================================

if steer6_df is not None:
    keys = [
        ("LOW_T2T5T10", 1, 1.0),
        ("LOW_T2T5T10", 2, 1.0),
        ("LOW_T2T5T10", 5, 1.0),
        ("LOW_T2T5T10", 10, 1.0),
        ("T5_ONLY", 1, 1.0),
        ("T10_ONLY", 1, 1.0),
        ("T10_ONLY", 10, 1.0),
    ]

    rows = []
    for mode, delta, alpha in keys:
        sub = steer6_df[
            (steer6_df["mode"] == mode) &
            (steer6_df["delta"] == delta) &
            (steer6_df["alpha"] == alpha)
        ]
        if len(sub) == 0:
            continue

        r = sub.iloc[0]
        rows.append({
            "mode": mode,
            "delta": int(delta),
            "alpha": alpha,
            "exact_original": pct(r["exact_original"]),
            "exact_delta": pct(r["exact_delta"]),
            "median_shift": num(r["median_shift"], 1),
            "same_units": pct(r["same_units"]),
            "follow10": pct(r["follow10"]),
            "stay10": pct(r["stay10"]),
            "net10": pct(r["net10"]),
            "follow5": pct(r["follow5"]),
            "stay5": pct(r["stay5"]),
            "net5": pct(r["net5"]),
            "follow2": pct(r["follow2"]),
            "stay2": pct(r["stay2"]),
            "net2": pct(r["net2"]),
            "human_read": (
                "Main residue steering: follow10 > stay10 means units digit was pushed."
                if mode == "LOW_T2T5T10" and delta == 1
                else "Null control: delta=10 should not change low-period residue."
                if delta == 10
                else ""
            ),
        })

    steer6_human = pd.DataFrame(rows)
    human_tables["08_low_period_residue_steering"] = steer6_human
    show("6. LOW-PERIOD RESIDUE STEERING", steer6_human)

    md("## 6. Low-period residue steering")
    md("")
    md("- This is sufficiency-style evidence for the residue channel.")
    md("- `follow10 > stay10` for delta=1 means the intervention pushed the output units digit toward the counterfactual.")
    md("- delta=10 is a null control for T2/T5/T10 because s and s+10 have the same mod2/mod5/mod10.")
    md("")

# ============================================================
# 7. MAGNITUDE STEERING
# ============================================================

if steer7_df is not None:
    keys = [
        ("LOW_T2T5T10", 10, 1.0),
        ("T100_ONLY", 10, 1.0),
        ("T100_ONLY", 20, 1.0),
        ("T100_ONLY", 50, 1.0),
        ("T100_LINEAR", 10, 1.0),
        ("T100_LINEAR", 20, 1.0),
        ("T100_LINEAR", 50, 1.0),
        ("FULL_T2T5T10T100_LINEAR", 10, 1.0),
        ("FULL_T2T5T10T100_LINEAR", 20, 1.0),
        ("LINEAR_ONLY", 50, 1.0),
    ]

    rows = []
    for mode, delta, alpha in keys:
        sub = steer7_df[
            (steer7_df["mode"] == mode) &
            (steer7_df["delta"] == delta) &
            (steer7_df["alpha"] == alpha)
        ]
        if len(sub) == 0:
            continue

        r = sub.iloc[0]
        rows.append({
            "mode": mode,
            "delta": int(delta),
            "alpha": alpha,
            "exact_original": pct(r["exact_original"]),
            "exact_delta": pct(r["exact_delta"]),
            "median_shift": num(r["median_shift"], 1),
            "same_units": pct(r["same_units"]),
            "same_decade": pct(r["same_decade"]),
            "follow100": pct(r["follow100"]),
            "stay100": pct(r["stay100"]),
            "net100": pct(r["net100"]),
            "human_read": (
                "Null control: low-period channel cannot distinguish s from s+10."
                if mode == "LOW_T2T5T10"
                else "Magnitude steering worked coarsely if median_shift ≈ delta and same_units stays high."
            ),
        })

    steer7_human = pd.DataFrame(rows)
    human_tables["09_t100_linear_magnitude_steering"] = steer7_human
    show("7. T100 / LINEAR MAGNITUDE STEERING", steer7_human)

    md("## 7. T100 / linear magnitude steering")
    md("")
    md("- This is noisier than ablation, so treat it as supporting evidence.")
    md("- `median_shift ≈ delta` with high `same_units` is the cleanest evidence of magnitude steering.")
    md("- `LOW_T2T5T10 delta=10` should be null-ish.")
    md("")


# ============================================================
# SAVE ALL HUMAN TABLES
# ============================================================

combined_rows = []
for name, df in human_tables.items():
    tmp = df.copy()
    tmp.insert(0, "table_name", name)
    combined_rows.append(tmp)

if combined_rows:
    combined = pd.concat(combined_rows, ignore_index=True, sort=False)
    save_table(combined, "human_summary_tables.csv")

md_path = os.path.join(OUT_DIR, "human_summary.md")
with open(md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(markdown_lines))

print("\n" + "=" * 100)
print("SAVED HUMAN SUMMARY")
print("=" * 100)
print("saved:", md_path)
print("saved:", os.path.join(OUT_DIR, "human_summary_tables.csv"))
print("\nOpen human_summary.md for the prose version.")

In [ ]:
# ============================================================
# TEST 8: SOURCE-TOKEN RESIDUE ARITHMETIC STEERING
#
# Purpose:
#   Test whether residue information on the source tokens a and b
#   is internally composed by modular addition.
#
# Core idea:
#   Pin the T-k residue channels on the a-token and b-token
#   to counterfactual residues, then check whether the model's
#   predicted answer follows:
#
#       pred mod k == (injected_a_residue + injected_b_residue) mod k
#
# This is stronger than answer-token steering:
#   - Test 6 asks whether the answer residue can be read out.
#   - Test 8 asks whether source-token residues are added inside the model.
#
# Main conditions:
#   SRC_T2_ONLY:
#       Inject mod2 source residues; check pred mod2.
#
#   SRC_T5_ONLY:
#       Inject mod5 source residues; check pred mod5.
#
#   SRC_T2T5:
#       Inject independent mod2 and mod5 residues on a and b.
#       Check whether pred mod2, mod5, and CRT-composed mod10 follow.
#
#   SRC_T2T5T10:
#       Same as SRC_T2T5, but also pins T10 consistently with the
#       CRT-composed source units digit. This is a stronger low-period
#       source intervention, less pure as an RNS test but useful.
#
# Saves:
#   test8_source_residue_arithmetic.csv
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.linear_model import Ridge

print("\n" + "=" * 100)
print("TEST 8: SOURCE-TOKEN RESIDUE ARITHMETIC STEERING")
print("=" * 100)

# ------------------------------------------------------------
# Required-global checks
# ------------------------------------------------------------

_required = [
    "model",
    "tokenizer",
    "baseline_correct_examples",
    "eval_examples",
    "layers",
    "L0",
    "BATCH_SIZE",
    "RIDGE_ALPHA",
    "pred_token_to_int",
    "save_df",
]

_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(f"Missing required globals for TEST 8: {_missing}")

TEST8_SEED = globals().get("SEED", 42) + 8000
TEST8_FIT_N = min(globals().get("FIT_N", 4000), len(baseline_correct_examples))
TEST8_MAX_EVAL = None  # set to e.g. 1000 for smoke test
TEST8_SOURCE_LAYERS = list(range(0, min(int(L0), int(model.config.num_hidden_layers))))

if TEST8_MAX_EVAL is None:
    TEST8_EVAL = eval_examples
else:
    TEST8_EVAL = eval_examples[:TEST8_MAX_EVAL]

print("fit examples:", TEST8_FIT_N)
print("eval examples:", len(TEST8_EVAL))
print("source layers:", TEST8_SOURCE_LAYERS)


# ------------------------------------------------------------
# Block compatibility: Pythia / GPT-J / Llama-style
# ------------------------------------------------------------

def get_block_list_test8():
    if "blocks" in globals():
        return blocks, "blocks"
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        return model.gpt_neox.layers, "model.gpt_neox.layers"
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h, "model.transformer.h"
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers, "model.model.layers"
    raise RuntimeError("Could not find transformer block list for TEST 8.")

TEST8_BLOCKS, TEST8_BLOCK_PATH = get_block_list_test8()
print("using block path:", TEST8_BLOCK_PATH)


# ------------------------------------------------------------
# Token-position helpers
# ------------------------------------------------------------

def _find_expr_spans(prompt, a, b):
    """
    Finds character spans for a and b inside the expression '{a}+{b}='.
    This avoids accidentally matching numbers in the instruction prefix.
    """
    a_s = str(int(a))
    b_s = str(int(b))
    expr = f"{a_s}+{b_s}="
    start = prompt.rfind(expr)

    if start < 0:
        # Fallback: find a, then b after a.
        a_start = prompt.rfind(a_s)
        if a_start < 0:
            raise ValueError(f"Could not find a={a_s} in prompt: {prompt!r}")
        b_start = prompt.find(b_s, a_start + len(a_s))
        if b_start < 0:
            raise ValueError(f"Could not find b={b_s} after a in prompt: {prompt!r}")
    else:
        a_start = start
        b_start = start + len(a_s) + 1

    a_span = (a_start, a_start + len(a_s))
    b_span = (b_start, b_start + len(b_s))
    return a_span, b_span


def _token_index_for_char_span(offsets, span):
    """
    Given tokenizer offset mappings for one sequence and a char span,
    return the token index with maximal overlap.
    """
    s0, s1 = span
    best_i = None
    best_overlap = 0

    for i, off in enumerate(offsets):
        t0, t1 = int(off[0]), int(off[1])

        # Skip padding/special offsets.
        if t0 == t1:
            continue

        overlap = max(0, min(t1, s1) - max(t0, s0))
        if overlap > best_overlap:
            best_overlap = overlap
            best_i = i

    if best_i is None or best_overlap <= 0:
        raise ValueError(f"Could not map char span {span} to token offset mapping.")

    return int(best_i)


def get_ab_token_positions(batch_rows):
    """
    Returns:
      enc: tokenizer batch encoding on model.device
      a_pos: tensor [B]
      b_pos: tensor [B]
    """
    prompts = [x["prompt"] for x in batch_rows]

    if not getattr(tokenizer, "is_fast", False):
        raise RuntimeError(
            "TEST 8 needs a fast tokenizer with offset_mapping for robust a/b token positions."
        )

    enc_cpu = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    offsets = enc_cpu.pop("offset_mapping").tolist()

    a_positions = []
    b_positions = []

    for i, ex in enumerate(batch_rows):
        a_span, b_span = _find_expr_spans(ex["prompt"], ex["a"], ex["b"])
        a_positions.append(_token_index_for_char_span(offsets[i], a_span))
        b_positions.append(_token_index_for_char_span(offsets[i], b_span))

    enc = {k: v.to(model.device) for k, v in enc_cpu.items()}
    a_pos = torch.tensor(a_positions, dtype=torch.long, device=model.device)
    b_pos = torch.tensor(b_positions, dtype=torch.long, device=model.device)

    return enc, a_pos, b_pos


# Quick sanity check on first few examples.
print("\nToken-position sanity check:")
for ex in TEST8_EVAL[:3]:
    enc_tmp, a_pos_tmp, b_pos_tmp = get_ab_token_positions([ex])
    ids = enc_tmp["input_ids"][0].detach().cpu().tolist()
    a_tok = tokenizer.decode([ids[int(a_pos_tmp.item())]])
    b_tok = tokenizer.decode([ids[int(b_pos_tmp.item())]])
    print(f"prompt={ex['prompt']!r} | a_tok={a_tok!r} | b_tok={b_tok!r}")


# ------------------------------------------------------------
# Basis / fitting helpers
# ------------------------------------------------------------

def make_period_basis_from_values(values, periods):
    """
    Basis from actual scalar values, e.g. a or b:
      [cos(2π value/T), sin(2π value/T)] for each T.
    """
    v = np.asarray(values, dtype=np.float64)
    cols = []

    for T in periods:
        theta = 2.0 * np.pi * v / float(T)
        cols.append(np.cos(theta))
        cols.append(np.sin(theta))

    if len(cols) == 0:
        raise ValueError("Empty period basis.")

    return np.column_stack(cols)


def make_period_basis_from_targets(target_by_period, periods):
    """
    Basis from counterfactual residues for each period.
    target_by_period maps T -> np.array shape [B].
    """
    cols = []

    for T in periods:
        if T not in target_by_period:
            raise ValueError(f"Missing target residues for period T={T}")

        r = np.asarray(target_by_period[T], dtype=np.float64)
        theta = 2.0 * np.pi * r / float(T)
        cols.append(np.cos(theta))
        cols.append(np.sin(theta))

    if len(cols) == 0:
        raise ValueError("Empty target basis.")

    return np.column_stack(cols)


def fit_QWB_source(X, values, periods):
    """
    Fit:
      X ≈ Y @ W.T + bias

    Returns:
      Q    : orthonormal basis of learned source residue span [D, K]
      W    : regression directions [D, K]
      bias : intercept [D]
    """
    Y = make_period_basis_from_values(values, periods=periods)

    reg = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
    reg.fit(Y, X)

    W = reg.coef_
    bias = reg.intercept_

    Q, _ = np.linalg.qr(W)

    return (
        torch.tensor(Q, dtype=torch.float16, device=model.device),
        torch.tensor(W, dtype=torch.float16, device=model.device),
        torch.tensor(bias, dtype=torch.float16, device=model.device),
    )


@torch.no_grad()
def collect_source_resids(rows, source_layers):
    """
    Collect residuals at a-token and b-token positions for each layer.
    """
    store_a = {l: [] for l in source_layers}
    store_b = {l: [] for l in source_layers}
    avals = []
    bvals = []

    buf = {}

    def make_hook(layer):
        def hook(module, inputs, output):
            hs = output[0] if isinstance(output, tuple) else output
            buf[layer] = hs.detach()
        return hook

    handles = [TEST8_BLOCKS[l].register_forward_hook(make_hook(l)) for l in source_layers]

    try:
        for start in tqdm(range(0, len(rows), BATCH_SIZE), desc="TEST 8 collect source residuals"):
            batch = rows[start:start + BATCH_SIZE]
            enc, a_pos, b_pos = get_ab_token_positions(batch)
            bi = torch.arange(len(batch), device=model.device)

            buf.clear()
            model(**enc, use_cache=False)

            for l in source_layers:
                if l not in buf:
                    raise RuntimeError(f"Missing hook output for layer {l}")
                store_a[l].append(buf[l][bi, a_pos].float().cpu().numpy())
                store_b[l].append(buf[l][bi, b_pos].float().cpu().numpy())

            avals.extend([int(x["a"]) for x in batch])
            bvals.extend([int(x["b"]) for x in batch])

    finally:
        for h in handles:
            h.remove()

    Xa = {l: np.concatenate(store_a[l], axis=0) for l in source_layers}
    Xb = {l: np.concatenate(store_b[l], axis=0) for l in source_layers}

    return Xa, Xb, np.array(avals, dtype=np.float64), np.array(bvals, dtype=np.float64)


print("\nCollecting source-token residuals...")
Xa_src, Xb_src, A_src, B_src = collect_source_resids(
    baseline_correct_examples[:TEST8_FIT_N],
    TEST8_SOURCE_LAYERS,
)
print("done collecting source residuals.")


# ------------------------------------------------------------
# CRT helpers and target construction
# ------------------------------------------------------------

def crt_mod2_mod5_to_mod10(r2, r5):
    """
    Return unique u in [0, 9] such that:
      u % 2 == r2
      u % 5 == r5
    """
    r2 = int(r2) % 2
    r5 = int(r5) % 5

    for u in range(10):
        if u % 2 == r2 and u % 5 == r5:
            return u

    raise RuntimeError("CRT composition failed.")


def build_test8_targets(rows, mode, seed):
    """
    Builds counterfactual source residues for a and b.

    Returns a dict containing:
      target_by_period_a
      target_by_period_b
      target_mod2
      target_mod5
      target_mod10
    """
    rng = np.random.default_rng(seed)
    n = len(rows)

    true_sum = np.array([int(x["sum"]) for x in rows], dtype=int)

    out = {
        "target_by_period_a": {},
        "target_by_period_b": {},
        "target_mod2": None,
        "target_mod5": None,
        "target_mod10": None,
    }

    if mode == "SRC_T2_ONLY":
        ra2 = rng.integers(0, 2, size=n)
        rb2 = rng.integers(0, 2, size=n)
        target2 = (ra2 + rb2) % 2

        # Force conflict with true parity where possible.
        same = target2 == (true_sum % 2)
        rb2[same] = 1 - rb2[same]
        target2 = (ra2 + rb2) % 2

        out["target_by_period_a"][2] = ra2
        out["target_by_period_b"][2] = rb2
        out["target_mod2"] = target2
        return out

    if mode == "SRC_T5_ONLY":
        ra5 = rng.integers(0, 5, size=n)
        rb5 = rng.integers(0, 5, size=n)
        target5 = (ra5 + rb5) % 5

        # Force conflict with true mod5 where possible.
        same = target5 == (true_sum % 5)
        rb5[same] = (rb5[same] + 1) % 5
        target5 = (ra5 + rb5) % 5

        out["target_by_period_a"][5] = ra5
        out["target_by_period_b"][5] = rb5
        out["target_mod5"] = target5
        return out

    if mode in ["SRC_T2T5", "SRC_T2T5T10"]:
        ra2 = rng.integers(0, 2, size=n)
        rb2 = rng.integers(0, 2, size=n)
        ra5 = rng.integers(0, 5, size=n)
        rb5 = rng.integers(0, 5, size=n)

        ua = np.array([crt_mod2_mod5_to_mod10(x, y) for x, y in zip(ra2, ra5)], dtype=int)
        ub = np.array([crt_mod2_mod5_to_mod10(x, y) for x, y in zip(rb2, rb5)], dtype=int)

        target10 = (ua + ub) % 10

        # Force conflict with true units digit where possible.
        same = target10 == (true_sum % 10)
        rb5[same] = (rb5[same] + 1) % 5

        ua = np.array([crt_mod2_mod5_to_mod10(x, y) for x, y in zip(ra2, ra5)], dtype=int)
        ub = np.array([crt_mod2_mod5_to_mod10(x, y) for x, y in zip(rb2, rb5)], dtype=int)

        target2 = (ra2 + rb2) % 2
        target5 = (ra5 + rb5) % 5
        target10 = (ua + ub) % 10

        out["target_by_period_a"][2] = ra2
        out["target_by_period_b"][2] = rb2
        out["target_by_period_a"][5] = ra5
        out["target_by_period_b"][5] = rb5

        if mode == "SRC_T2T5T10":
            out["target_by_period_a"][10] = ua
            out["target_by_period_b"][10] = ub

        out["target_mod2"] = target2
        out["target_mod5"] = target5
        out["target_mod10"] = target10
        return out

    raise ValueError(f"Unknown TEST 8 mode: {mode}")


# ------------------------------------------------------------
# Source-token pin-steering hook
# ------------------------------------------------------------

_test8_a_pos = None
_test8_b_pos = None
_test8_bi = None
_test8_QA = {}
_test8_QB = {}
_test8_target_A = {}
_test8_target_B = {}

def make_test8_source_pin_hook(layer):
    def hook(module, inputs, output):
        global _test8_a_pos, _test8_b_pos, _test8_bi
        global _test8_QA, _test8_QB, _test8_target_A, _test8_target_B

        if isinstance(output, tuple):
            hs = output[0].clone()
            rest = tuple(output[1:])
        else:
            hs = output.clone()
            rest = None

        QA = _test8_QA[layer]
        QB = _test8_QB[layer]

        va = hs[_test8_bi, _test8_a_pos]
        vb = hs[_test8_bi, _test8_b_pos]

        hs[_test8_bi, _test8_a_pos] = va - (va @ QA) @ QA.T + _test8_target_A[layer]
        hs[_test8_bi, _test8_b_pos] = vb - (vb @ QB) @ QB.T + _test8_target_B[layer]

        return hs if rest is None else (hs,) + rest

    return hook


@torch.no_grad()
def eval_source_residue_steering(rows, QWB_A_by_layer, QWB_B_by_layer, periods, mode, seed):
    """
    Evaluate source-token residue steering.

    Main follow metrics:
      follow2  : pred % 2  == injected modular sum mod2
      follow5  : pred % 5  == injected modular sum mod5
      follow10 : pred % 10 == CRT-composed injected unit digit
    """
    global _test8_a_pos, _test8_b_pos, _test8_bi
    global _test8_QA, _test8_QB, _test8_target_A, _test8_target_B

    targets_all = build_test8_targets(rows, mode=mode, seed=seed)

    handles = [
        TEST8_BLOCKS[l].register_forward_hook(make_test8_source_pin_hook(l))
        for l in QWB_A_by_layer
    ]

    n = 0
    parseable = 0
    exact_original = 0

    follow2 = stay2 = 0
    follow5 = stay5 = 0
    follow10 = stay10 = 0

    pred_records = []

    try:
        for start in tqdm(range(0, len(rows), BATCH_SIZE), desc=f"TEST 8 eval {mode}"):
            batch = rows[start:start + BATCH_SIZE]
            batch_slice = slice(start, start + len(batch))

            enc, a_pos, b_pos = get_ab_token_positions(batch)
            bi = torch.arange(len(batch), device=model.device)

            _test8_a_pos = a_pos
            _test8_b_pos = b_pos
            _test8_bi = bi

            _test8_QA = {}
            _test8_QB = {}
            _test8_target_A = {}
            _test8_target_B = {}

            # Build target fitted content for this batch/layer.
            target_by_period_a_batch = {
                T: arr[batch_slice] for T, arr in targets_all["target_by_period_a"].items()
            }
            target_by_period_b_batch = {
                T: arr[batch_slice] for T, arr in targets_all["target_by_period_b"].items()
            }

            YA_np = make_period_basis_from_targets(target_by_period_a_batch, periods=periods)
            YB_np = make_period_basis_from_targets(target_by_period_b_batch, periods=periods)

            YA = torch.tensor(YA_np, dtype=torch.float16, device=model.device)
            YB = torch.tensor(YB_np, dtype=torch.float16, device=model.device)

            for l in QWB_A_by_layer:
                QA, WA, biasA = QWB_A_by_layer[l]
                QB, WB, biasB = QWB_B_by_layer[l]

                yhatA = YA @ WA.T + biasA
                yhatB = YB @ WB.T + biasB

                _test8_QA[l] = QA
                _test8_QB[l] = QB
                _test8_target_A[l] = (yhatA @ QA) @ QA.T
                _test8_target_B[l] = (yhatB @ QB) @ QB.T

            logits = model(**enc, use_cache=False).logits[bi, enc["attention_mask"].sum(dim=1) - 1]
            pred_ids = logits.argmax(dim=-1).detach().cpu().tolist()

            target2_batch = (
                targets_all["target_mod2"][batch_slice]
                if targets_all["target_mod2"] is not None else None
            )
            target5_batch = (
                targets_all["target_mod5"][batch_slice]
                if targets_all["target_mod5"] is not None else None
            )
            target10_batch = (
                targets_all["target_mod10"][batch_slice]
                if targets_all["target_mod10"] is not None else None
            )

            for j, (pid, ex) in enumerate(zip(pred_ids, batch)):
                n += 1
                exact_original += int(int(pid) == int(ex["target_token_id"]))

                pn = pred_token_to_int(pid)
                if pn is None:
                    pred_records.append({
                        "mode": mode,
                        "a": ex["a"],
                        "b": ex["b"],
                        "sum": ex["sum"],
                        "pred": None,
                        "parseable": 0,
                    })
                    continue

                parseable += 1
                true_sum = int(ex["sum"])

                rec = {
                    "mode": mode,
                    "a": ex["a"],
                    "b": ex["b"],
                    "sum": true_sum,
                    "pred": int(pn),
                    "parseable": 1,
                    "pred_mod2": int(pn % 2),
                    "pred_mod5": int(pn % 5),
                    "pred_mod10": int(pn % 10),
                    "true_mod2": int(true_sum % 2),
                    "true_mod5": int(true_sum % 5),
                    "true_mod10": int(true_sum % 10),
                }

                if target2_batch is not None:
                    t2 = int(target2_batch[j])
                    follow2 += int(pn % 2 == t2)
                    stay2 += int(pn % 2 == true_sum % 2)
                    rec["target_mod2"] = t2
                    rec["follow2"] = int(pn % 2 == t2)
                    rec["stay2"] = int(pn % 2 == true_sum % 2)

                if target5_batch is not None:
                    t5 = int(target5_batch[j])
                    follow5 += int(pn % 5 == t5)
                    stay5 += int(pn % 5 == true_sum % 5)
                    rec["target_mod5"] = t5
                    rec["follow5"] = int(pn % 5 == t5)
                    rec["stay5"] = int(pn % 5 == true_sum % 5)

                if target10_batch is not None:
                    t10 = int(target10_batch[j])
                    follow10 += int(pn % 10 == t10)
                    stay10 += int(pn % 10 == true_sum % 10)
                    rec["target_mod10"] = t10
                    rec["follow10"] = int(pn % 10 == t10)
                    rec["stay10"] = int(pn % 10 == true_sum % 10)

                pred_records.append(rec)

    finally:
        for h in handles:
            h.remove()

    n_safe = max(n, 1)

    out = {
        "mode": mode,
        "periods": str(periods),
        "n": n,
        "parseable": parseable / n_safe,
        "exact_original": exact_original / n_safe,
    }

    if targets_all["target_mod2"] is not None:
        out["follow2"] = follow2 / n_safe
        out["stay2"] = stay2 / n_safe
        out["net2"] = (follow2 - stay2) / n_safe

    if targets_all["target_mod5"] is not None:
        out["follow5"] = follow5 / n_safe
        out["stay5"] = stay5 / n_safe
        out["net5"] = (follow5 - stay5) / n_safe

    if targets_all["target_mod10"] is not None:
        out["follow10"] = follow10 / n_safe
        out["stay10"] = stay10 / n_safe
        out["net10"] = (follow10 - stay10) / n_safe

    return out, pred_records


# ------------------------------------------------------------
# Fit source spans and run conditions
# ------------------------------------------------------------

TEST8_CONDITIONS = [
    ("SRC_T2_ONLY", [2]),
    ("SRC_T5_ONLY", [5]),
    ("SRC_T2T5", [2, 5]),
    ("SRC_T2T5T10", [2, 5, 10]),
]

test8_rows = []
test8_pred_rows = []

for mode, periods in TEST8_CONDITIONS:
    print("\n" + "-" * 100)
    print(f"Fitting source spans for {mode}: periods={periods}")
    print("-" * 100)

    QWB_A = {
        l: fit_QWB_source(Xa_src[l], A_src, periods=periods)
        for l in TEST8_SOURCE_LAYERS
    }
    QWB_B = {
        l: fit_QWB_source(Xb_src[l], B_src, periods=periods)
        for l in TEST8_SOURCE_LAYERS
    }

    r, pred_records = eval_source_residue_steering(
        TEST8_EVAL,
        QWB_A,
        QWB_B,
        periods=periods,
        mode=mode,
        seed=TEST8_SEED + len(test8_rows),
    )

    test8_rows.append(r)
    test8_pred_rows.extend(pred_records)

    print("result:", r)

test8_df = pd.DataFrame(test8_rows)
test8_pred_df = pd.DataFrame(test8_pred_rows)

save_df(test8_df, "test8_source_residue_arithmetic.csv")
save_df(test8_pred_df, "test8_source_residue_arithmetic_predictions.csv")

print("\n" + "=" * 100)
print("TEST 8 SUMMARY")
print("=" * 100)
print(test8_df.to_string(index=False))
print("\nTEST 8 complete.")

In [ ]:
# ============================================================
# TEST 8 HUMAN-READABLE DASHBOARD
#
# Reads:
#   test8_source_residue_arithmetic.csv
#
# Prints:
#   source-token residue arithmetic summary
# ============================================================

import os
import pandas as pd
import numpy as np

print("\n" + "=" * 100)
print("TEST 8 HUMAN SUMMARY — SOURCE-TOKEN RESIDUE ARITHMETIC")
print("=" * 100)

def pct8(x):
    if pd.isna(x):
        return ""
    return f"{100 * float(x):.1f}%"

path8 = os.path.join(OUT_DIR, "test8_source_residue_arithmetic.csv")

if not os.path.exists(path8):
    print("missing:", path8)
else:
    df8 = pd.read_csv(path8)
    print(f"loaded: {path8} ({len(df8)} rows)")

    rows = []

    for _, r in df8.iterrows():
        mode = r["mode"]

        human_read = ""

        if mode == "SRC_T2_ONLY":
            human_read = (
                "Source mod2 test: pred parity should follow injected "
                "(a_mod2 + b_mod2) mod 2."
            )
        elif mode == "SRC_T5_ONLY":
            human_read = (
                "Source mod5 test: pred mod5 should follow injected "
                "(a_mod5 + b_mod5) mod 5."
            )
        elif mode == "SRC_T2T5":
            human_read = (
                "Pure CRT-like/RNS test: independently injected mod2 and mod5 "
                "source residues should compose into pred mod10."
            )
        elif mode == "SRC_T2T5T10":
            human_read = (
                "Full low-period source test: mod2/mod5 plus consistent T10 source units."
            )

        row = {
            "mode": mode,
            "periods": r.get("periods", ""),
            "n": int(r["n"]),
            "parseable": pct8(r.get("parseable")),
            "exact_original": pct8(r.get("exact_original")),
            "human_read": human_read,
        }

        for k in [2, 5, 10]:
            if f"follow{k}" in df8.columns and not pd.isna(r.get(f"follow{k}", np.nan)):
                row[f"follow{k}"] = pct8(r.get(f"follow{k}"))
                row[f"stay{k}"] = pct8(r.get(f"stay{k}"))
                row[f"net{k}"] = pct8(r.get(f"net{k}"))

        rows.append(row)

    human8 = pd.DataFrame(rows)

    # Put columns in a readable order.
    preferred_cols = [
        "mode", "periods", "n", "parseable", "exact_original",
        "follow2", "stay2", "net2",
        "follow5", "stay5", "net5",
        "follow10", "stay10", "net10",
        "human_read",
    ]
    preferred_cols = [c for c in preferred_cols if c in human8.columns]
    human8 = human8[preferred_cols]

    print(human8.to_string(index=False))

    out_path8 = os.path.join(OUT_DIR, "test8_source_residue_arithmetic_human.csv")
    human8.to_csv(out_path8, index=False)
    print("\nsaved:", out_path8)

    print("\nHow to read:")
    print("- follow{k}: predicted answer residue matches the injected source-residue sum mod k.")
    print("- stay{k}: predicted answer residue stays at the original true a+b residue mod k.")
    print("- net{k} = follow{k} - stay{k}. Large positive net means source steering overrode the original residue.")
    print("- The key RNS/CRT-like row is SRC_T2T5: follow2, follow5, and especially follow10.")